Instructions:
- compartments_lowercase may handle compartments differently for your model, so consider updating the way compartments are processed
- Update "proteome customization" section to suit your organism

# initialize

In [ ]:

import cobra,json,os,sys,re,pandas as pd, io, requests
from bs4 import BeautifulSoup
from collections import OrderedDict
from copy import deepcopy

# if running from another notebook, retain the 'decompartmentalize' variable if it exists; otherwise, set to False (so that decompartmentalization is only performed if explicitly specified in the notebook)
decompartmentalize = sys.modules['__main__'].__dict__.get('decompartmentalize', False)
load_model = False if decompartmentalize else True

# keep going up until the "pycore" folder is found
wd = os.getcwd()
while not os.path.exists('pycore'):
	os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *
SM_path = SM_path if not decompartmentalize else sys.modules['__main__'].__dict__.get('SM_path_decomp', None)

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [ ]:
# proteome customization
ignore_MW_zeros = True # whether to ignore enzyme complexes with a MW of 0 (True recommended, as these may stem from duplicate genes in the GSM model; if some rxns are missing enzymes, consider setting to False)
# all proteins to include in your model (could be an organism ID, proteome ID, etc.; check to ensure your query is correct)
proteome = RBA_defaults.proteome # UniProt proteome ID for your organism
# any additional accessions you want to add to the model (these will be added to the proteome query)

extra_accessions = RBA_defaults.extra_proteins  # list of UniProt accessions to include in the proteome query
## Ideally, include UniProt IDs from your proteomics data here, to curate all proteins at once (see example below)
## can also determine extra accessions from proteomics data spreadsheet (commented example below)
# extra_accessions.extend([i for i in read_spreadsheet('~/Documents/pkRBA/parameterization/kapp/datasets/raw_data_files/ShenEtAl2024_multiomics_41589_2024_1571_MOESM3_ESM.xlsx', sheet_name='Table 10c. abs_prot_IO_SD108')['Entry']])

# extra accessions processed later, to reduce the risk of length limits in the URL
url = build_uniprot_query(proteome_id=proteome)

In [3]:
old_model = load_cobra_model(SM_path) if not decompartmentalize else deepcopy(model)
# sort metabolites and reactions by id
# old_model.metabolites = sorted(old_model.metabolites, key=lambda x: x.id)
# old_model.reactions = sorted(old_model.reactions, key=lambda x: x.id)
# save the model
# save_cobra_model(old_model, model_path)

In [4]:
def get_protein_id(gene):
	"""
	Get the protein ID from a COBRApy gene object.
	"""
	# Check if the gene object has the attribute
	try:
		# If the attribute exists, return its value
		protein_id = eval(f"gene.{RBA_defaults.protein_id_in_gene_format}")  # get the protein ID from the gene object
		return protein_id
	except (KeyError, AttributeError) as e:
		# If the attribute does not exist, return None or handle it as needed
		# print(f"Attribute '{RBA_defaults.protein_id_in_gene_format}' not found in gene {gene.id}.")
		# You can also raise an exception or return a default value if needed
		# raise AttributeError(f"Attribute '{RBA_defaults.protein_id_in_gene_format}' not found in gene object.")
		return None
	# if hasattr(gene, RBA_defaults.protein_id_in_gene_format):
	# 	# If the attribute exists, return its value
	# 	return eval(f"gene.{RBA_defaults.protein_id_in_gene_format}")  # get the protein ID from the gene object
	# else:
	# 	# If the attribute does not exist, return None or handle it as needed
	# 	print(f"Attribute '{RBA_defaults.protein_id_in_gene_format}' not found in gene {gene.id}.")
	# 	# You can also raise an exception or return a default value if needed
	# 	# raise AttributeError(f"Attribute '{RBA_defaults.protein_id_in_gene_format}' not found in gene object.")

warnings_summary = [] # msgs to print later
def warn(msg,ignore_duplicate_warnings=True):
	"""Append a warning message to the warnings list (raises warnings at the end of the script) and print it (for easier debugging)."""
	if ignore_duplicate_warnings and msg in warnings_summary:
		return
	warnings_summary.append(msg)
	print("Warning:", msg)
genes_without_protein_id = []
gene_protein_mapping = {}
for gene in old_model.genes:
	prot = get_protein_id(gene)
	# if the gene doesn't have the attribute, print a warning
	if prot in (None,[]):
		genes_without_protein_id.append(gene.id)
		# print(f"{gene.id} does not have the attribute '{RBA_defaults.protein_id_in_gene_format}'")
	elif prot not in extra_accessions:
		extra_accessions.append(prot)
	# if protein already in gene_protein_mapping values, append to the list
	elif prot in sum(gene_protein_mapping.values(),[]):
		warn(f"Protein ID {prot} is associated with multiple genes (including {gene.id}). Remove duplicates to avoid issues.")
	gene_protein_mapping[gene.id] = [prot]
if genes_without_protein_id:
	print(f"Warning: {len(genes_without_protein_id)} genes do not have the attribute '{RBA_defaults.protein_id_in_gene_format}': {genes_without_protein_id}")

In [5]:
# regex = {'compartment':re.compile(r'^.+_(.+)$') # last underscore denotes compartment
		#  }
regex = {'compartment':re.compile(r'^(.+)_(.+)$') # last underscore denotes compartment
		 }

In [6]:
def extract_details_from_rxnid_SM(met):
	if met.compartment:
		regex_match = regex['compartment'].match(met.id)
		if regex_match.group(2) == met.compartment:
			return regex_match.group(1)
extract_details_from_rxnid_SM(old_model.metabolites.o2_e)
# make the new method available to cobra.metabolite.Metabolite
# cobra.core.metabolite.Metabolite.base_id = str(extract_details_from_rxnid_SM)
# cobra.core.metabolite.Metabolite.id_without_compartment = id_without_compartment
# type(old_model.metabolites.o2_e)

'o2'

In [7]:
def parse_reaction(reaction):
	# use regex['compartment'] to parse the reaction
	m = regex['compartment'].match(reaction)
	if m:
		# print the match
		print(m.group(1))
	else:
		print("No match found")
parse_reaction('xt_c_c0')

xt_c


In [8]:
# x = 'y{1+1}'
# eval(f'f"{x}"'), x
rxn = old_model.reactions.EX_o2_e
rxn = old_model.reactions.O2t_c_e
def rxn_id_format(rxn):
	# get the reaction name
	rxn_name = rxn.name
	# get the compartment
	m = regex['compartment'].match(rxn.id)
	if m:
		compartment = m.group(1)
		# replace the compartment with the compartment name
		rxn_name = rxn_name.replace(f'_{compartment}','')
	# return the formatted reaction name
	return rxn_name
rxn.compartments
# x

{'c', 'e'}

In [9]:
# create version of model condensing all compartments into one
def dict_from_metabolite(metabolite):
	"""Convert a cobra.Metabolite object to a dictionary, excluding private attributes."""
	import re
	metabolite_dict = {re.sub(r'^_(id|annotation)$', r'\1', key): value for key, value in metabolite.__dict__.items() if key not in ['_model', '_reaction','_bound']}
	return metabolite_dict
def unique_metabolites(cobra_model,diff_attr='version_diffs',match_logic="\met.formula and \met.charge and (\met.annotation['metanetx.chemical'] or \met.annotation['chebi'] or \met.annotation['pubchem.compound'] or \met.annotation['seed.compound'] or \met.annotation['biocyc'])"):
	"""Find all metabolites that represent unique compounds.

	**diff_attr**: str with name of attribute in model.metabolites that will store differences between metabolites deemed similar.
	
	**match_logic**: str with logical expression indicating how attributes will affect whether 2 metabolites are identical. 
		Write attributes prefaced by "\met" (representing "model.metabolites"), e.g. "\met.formula" or "\met.annotation['chebi']']"
		Operators include 'and', 'or', 'not'. Use parentheses to group expressions.

	
	Output: list with metabolites in JSON form.
	"""
	model = deepcopy(cobra_model)
	# obtain all types of attributes from the model's metabolites, to see if notation conflicts with match_logic format
	metabolite_attributes = set()
	for metabolite in model.metabolites:
		for key in metabolite.__dict__.keys():
			metabolite_attributes.add(key)
	logic_gates = [' or ', ' and ', ' not ']
	issues = []
	for metabolite_attribute in metabolite_attributes:
		for gate in logic_gates:
			if gate in metabolite_attribute:
				issues.append((metabolite_attribute, gate))
	# if any metabolite attributes contain logic words, raise an error
	if issues:
		raise ValueError(f"Attributes in match_logic must be in the format '\\met.attribute', but the following metabolite attributes overlap with logic gates in match_logic (either change the attributes or logic gate formatting): {issues}")
	
	# extract all variables from match_logic, starting with \met. and ending with the first logic gate; remove any closed parentheses at the end
	# split match_logic by logic gates to get all variables
	variables = set([re.sub(r'.*\\met.|\)$', '', v) for v in re.split(r'{}'.format('|'.join(logic_gates)), match_logic)])
	match_logic_simple = re.sub(r'\\met.', '', match_logic)
	
	unique_mets = [model.metabolites[0]] # start with 1st metabolite to streamline future comparisons
	for met in model.metabolites[1:]:
		is_unique = True # assume metabolite is unique until proven otherwise
		met2_index = 0
		for met2 in unique_mets:
			met2_index += 1
			# initialize all variables to None
			logic_values = {var: False for var in variables}
			# check if each variable (if present) is the same for both metabolites
			# print(met.id, met2)
			for var in variables:
				# only make logic_values[var] True if it's identical for both metabolites or if it's not present in one or both metabolites
				if hasattr(met, var) and hasattr(met2, var):
					if getattr(met, var, '') == getattr(met2, var, ''):
						# print(f'{var} is the same for {met.id} and {met2.id}')
						logic_values[var] = True
					# else:
					# 	print(f'{var} is different for {met.id} and {met2.id}: {getattr(met, var)} vs {getattr(met2, var)}')
				elif hasattr(met, var) or hasattr(met2, var):
					# print(f'{var} not present in {met.id} or {met2.id}')
					logic_values[var] = True
				else:
					# print(f'{var} is not present in {met.id} or {met2.id}')
					logic_values[var] = True
					pass
			# evaluate match_logic for met and met2
			expr = match_logic_simple
			for var in sorted(variables, key=len, reverse=True):
				expr = expr.replace(var, str(logic_values[var]))
			if eval(expr):
				print(f'{met.id} and {met2.id} are the same')
				is_unique = False
				break
		if is_unique:
			unique_mets.append(met)
		else:
			# find differences between metabolites
			diffs = {met.id: {}}
			for attr in dict_from_metabolite(met).keys():
				if attr not in ['id']: # filtering out ID since it's already used as key
					if getattr(met, attr, '') != getattr(met2, attr, ''):
						diffs[met.id][attr] = getattr(met, attr, '')
			# store differences in the diff_attr attribute within unique_mets
			# setattr(unique_mets[-1], diff_attr, diffs)
			# setattr(met2, diff_attr, diffs)

			# unique_mets[met2_index-1].setdefault(diff_attr, {}).update(diffs)
			# unique_mets[met2_index-1].version_diffs.setdefault(met.id, {}).update(diffs)
			if diff_attr not in unique_mets[met2_index-1].__dict__.keys():
				setattr(unique_mets[met2_index-1], diff_attr, {})
			unique_mets[met2_index-1].version_diffs.update(diffs)
			# setattr(unique_mets[met2_index-1], diff_attr, diffs)
	# remove all mets from model not in unique_mets
	model.metabolites = unique_mets
	return unique_mets, model

unique_mets, model_simplified = unique_metabolites(old_model)
# make dict version
unique_mets_dict = [dict_from_metabolite(met) for met in unique_mets]
mets_dict = [dict_from_metabolite(met) for met in old_model.metabolites]
# save to file
with open('./input/unique_mets.json', 'w') as f:
	json.dump(unique_mets_dict, f, indent=2)
with open('./input/mets.json', 'w') as f:
	json.dump(mets_dict, f, indent=2, sort_keys=True)

def make_model_with_single_compartment(model):
	"""Combine all metabolites and/or reactions in different compartments into one compartment, for easier updating and analysis.
	
	Output: JSON file with all compartments written as lists under "notes" as "all_compartments"
	"""
	
	model = deepcopy(model)
	if len(model.compartments) == 1:
		print('Model already has only one compartment')
		return model
	else:
		# combine all compartments into one
		compartment = 'c'
		for metabolite in model.metabolites:
			metabolite.compartment = compartment
		for reaction in model.reactions:
			reaction.compartment = compartment
		# for all mets with differences, make ["notes"]["all_compartments"] with lists of differences
		# how to deal w/ rxns involving multiple compartments? Make ["notes"]["all_compartments"] a dict with all compartment-specific names as keys and lists of compartment-specific names as values

	# find all metabolites 
	return model

nad_n and nad_r are the same
h_mm and h_g are the same
amet_e and amet_n are the same
2mbald_m and 3mbald_e are the same
dhap_rm and dhap_m are the same
fecost_e and fecost_c are the same
2mbac_e and iamac_c are the same
pi_rm and pi_m are the same
h2o_l and h2o_g are the same
quln_m and quln_c are the same
pi_e and pi_m are the same
g1p_c and f6p_c are the same
co2_n and co2_e are the same
fru_c and glc__D_e are the same
ile__L_e and ile__L_m are the same
hxccoa_c and hxccoa_x are the same
glutrna_c and glutrna_m are the same
o2_rm and o2_e are the same
udpgal_g and udpg_c are the same
leu__L_m and ile__L_m are the same
hxccoa_rm and hxccoa_x are the same
co2_r and co2_e are the same
inost_e and glc__D_e are the same
dhnpt_m and dhnpt_c are the same
asn__L_c and asn__L_m are the same
amp_n and amp_l are the same
cer2B_26_r and cer2A_26_m are the same
h2o_e and h2o_g are the same
id3acald_c and id3acald_m are the same
amet_rm and amet_n are the same
2phetoh_m and 2phetoh_e are the same

In [10]:
# sort metabolites by ID
model_simplified.metabolites = sorted(model_simplified.metabolites, key=lambda x: x.id)

In [11]:
def expand_metabolites(met_list,diff_attr='version_diffs'):
	"""Expand all metabolites in a list to include differences in attributes.
	
	Output: list with metabolites in JSON form.
	"""
	expanded_mets = []
	for met in met_list.copy():
		# if met has differences, expand it
		if hasattr(met, diff_attr):
			# get all differences
			diffs = getattr(met, diff_attr)
			# for each difference, create a new metabolite with that difference
			for new_met_id,diff in diffs.items():
				# print(new_met_id,diff)
				# create a new metabolite with the difference
				new_met = deepcopy(met)
				new_met.id = new_met_id
				for attr, value in diff.items():
					setattr(new_met, attr, value)
				expanded_mets.append(new_met)
				delattr(new_met, diff_attr)
		# remove the diff_attr attribute from the metabolite
		if hasattr(met, diff_attr):
			delattr(met, diff_attr)
		expanded_mets.append(met)
	return expanded_mets
expanded_mets = expand_metabolites(unique_mets)
# make dict version
expanded_mets_dict = [dict_from_metabolite(met) for met in expanded_mets]
# save to file
with open('./input/expanded_mets.json', 'w') as f:
	json.dump(expanded_mets_dict, f, indent=2, sort_keys=True)

In [12]:
# find which mets aren't in the expanded list but are in old_model
expanded_met_ids = [met.id for met in expanded_mets]
mets_not_expanded = [met.id for met in old_model.metabolites if met.id not in expanded_met_ids]
mets_not_expanded
# make dict version
# mets_not_expanded_dict = [dict_from_metabolite(met) for met in mets_not_expanded]
# save to file
# with open('./input/mets_not_expanded.json', 'w') as f:
# 	json.dump(mets_not_expanded_dict, f, indent=2)

[]

In [13]:
model_reduced = None
model_reduced_chebi_ids = []
if os.path.exists('_reduced'.join(os.path.splitext(SM_path))):
	model_reduced = load_cobra_model('_reduced'.join(os.path.splitext(SM_path)))
	model_reduced_chebi_ids = [m.annotation.get('chebi','').replace('CHEBI:','') for m in model_reduced.metabolites if 'chebi' in m.annotation]

In [14]:
# with open('~/Downloads/bigg_mets.json', 'r') as f:
# 	bigg = json.load(f)

# with open('./input/bigg_mets.json', 'w') as f:
# 	json.dump(bigg, f, indent=4)

# mapping metabolites to ChEBI IDs

In [15]:
# map IDs to CHEBI numbers, and search web for missing IDs (can take a few mins)
model = deepcopy(old_model)
met_chebi_map = {}
chebi_ids_ambiguous = set()
ids_found = set()

In [ ]:
# check if any metabolites have a kegg annotation

isotope_regex_chebi = '[A-Z][a-z]*|\[\d*[A-Z][a-z]*' # leave bracket in to better distinguish isotopes from coefficients
def compare_metabolite_to_database(met, database_formulas, database_charges, allow_formula_adjustment=False):
	"""Compare a metabolite's formula and charge to a database's formulas and charges.
	
	Output: True if the metabolite matches any of the database formulas and charges, False otherwise.

	allow_formula_adjustment: whether to allow adjustment of H atoms in formula comparison (to account for protonation state differences).
	"""
	# if met.formula isn't empty and database_formulas is not empty, check if met.formula is in database_formulas
	if not (met.formula and database_formulas):
		return False
	compiled_elements = compile_elements_from_formula(met.formula,sort_elements=True,element_regex=isotope_regex_chebi)
	formulae_compiled = [compile_elements_from_formula(f,sort_elements=True,element_regex=isotope_regex_chebi) for f in database_formulas]
	charges_match = False
	if met.charge is not None and database_charges:
		if met.charge in database_charges:
			charges_match = True
		else:
			charges_match = False
	elif met.charge is None or not database_charges or database_charges == [None]:
		charges_match = True
	if compiled_elements in formulae_compiled: 
		if charges_match or not database_charges:
			return True
		else:
			warn(f'Charge mismatch for {met.id}: model has charge {met.charge}, database has charges {database_charges}',ignore_duplicate_warnings=True)
			return False
	if not allow_formula_adjustment:
		return False

	# check if any of the formulae match when converting non-standard elements to R or X, since BIGG sometimes uses these but the model may have a different notation
	compiled_elements_simplified = compile_elements_from_formula(met.formula,sort_elements=True, nonelement_rule='R', element_regex=isotope_regex_chebi)
	match_found = False
	for f in database_formulas:
		compiled_f = compile_elements_from_formula(f,sort_elements=True, nonelement_rule='R', element_regex=isotope_regex_chebi)
		if not charges_match:
			diff_H = compiled_elements.get('H',0) - compiled_f.get('H',0)
			compiled_f['H'] = compiled_f.get('H',0) + diff_H
			# if charge doesn't match when adjusting for H atoms, continue
			if not (met.charge is not None and database_charges and (met.charge - diff_H) in database_charges):
				continue
		if compiled_f == compiled_elements_simplified:
			match_found = True
			break
	if match_found:
		return True
	warn(f'Formula mismatch for {met.id}: model has {met.formula}, database has {database_formulas}',ignore_duplicate_warnings=True)
	return False

mets_to_check = [i for i in model.metabolites]
for met in mets_to_check:
	base_id = met.id.rsplit('_'+met.compartment,1)[0]
	if 'kegg.compound' in met.annotation:
		ids_found.add('kegg.compound')
	chebi_links = set()
	if 'chebi' in met.annotation:
		ids_found.add('chebi')
		num = met.annotation['chebi'].replace('CHEBI:','')
		chebi_links.add('CHEBI:'+num)
		# turn back on to speed up processing; turn off to validate existing chebi annotations
		if num != '':
			met_chebi_map.setdefault(met.annotation['chebi'].replace('CHEBI:',''),{'default':base_id})[met.compartment] = met.id
			# if there are multiple atoms in its formula, skip searching for chebi number. Focus on 1-element metabolites since these sometimes have incorrect chebi annotations in models
			if len(compile_elements_from_formula(met.formula,element_regex=isotope_regex_chebi).keys()) > 1 or met.formula is None or sum(compile_elements_from_formula(met.formula,element_regex=isotope_regex_chebi).values()) > 1:
				continue
	# try to find the chebi number from the name
	match_found = False
	potential_matches_to_review = []
	# see if met's base_id is already in met_chebi_map, and if so, compare formulas and charges
	for chebi_id, compartments in met_chebi_map.items():
		if compartments.get('default','') == base_id:
			# compare formulas and charges to ensure match
			# skip if the compartment already has a metabolite assigned
			if compartments.get(met.compartment,'') == met.id:
				continue
			for comp in compartments.keys():
				if comp == 'default':
					continue
				met2 = model.metabolites.get_by_id(compartments[comp])
				# &apos; represents apostrophe in XML/HTML. Some models may use both notations inconsistently
				if met2.name.replace("&apos;","'") != met.name.replace("&apos;","'"): # may lead to false negatives if names differ (e.g., compartment included in names), but safer than false positives
					print(f'Name mismatch when checking {met.id} against {met2.id} for CHEBI ID {chebi_id}: {met.name} vs {met2.name}')
					continue
				# compare formulas and charges to ensure match
				perfect_match = compare_metabolite_to_database(met2,
															[met.formula],
															[met.charge])
				if perfect_match:
					met_chebi_map.setdefault(chebi_id,{'default':base_id})[met.compartment] = met.id
					# add the annotation to the model
					met.annotation['chebi'] = 'CHEBI:'+chebi_id
					match_found = True
					break
	data = {}
	try:
		response = requests.get('http://bigg.ucsd.edu/api/v2/universal/metabolites/'+base_id, timeout=30)
		# if the request wasn't successful
		if response.status_code != 200:
			continue
		# get the json data
		data = response.json()
	except (ConnectionError, OSError, requests.exceptions.RequestException) as e:
		warn(e,ignore_duplicate_warnings=True)
	if 'database_links' not in data:
		data['database_links'] = {}
	# convert lists to sets for easier checking
	for key in data['database_links'].keys():
		data['database_links'][key] = set(i['id'] for i in data['database_links'][key])
	for annotation, database_link in (('metanetx.chemical','MetaNetX (MNX) Chemical'),
										('kegg.compound','KEGG Compound'),
										('seed.compound','SEED Compound'),
										('biocyc','BioCyc'),
										('pubchem.compound','PubChem Compound')):
		if database_link in data['database_links']:
			# should manually review these annotations if adding them later, since not all may be correct (e.g., deprotonated form of an acid)
			# if annotation not in met.annotation:
			# 	met.annotation[annotation] = data['database_links'][database_link][0]['id']
			ids_found.add(annotation)
		else:
			data['database_links'][database_link] = set()
		if annotation in met.annotation:
			# add the metabolite annotation to the database link if not already present
			if met.annotation[annotation] not in data['database_links'][database_link]:
				data['database_links'][database_link].add(met.annotation[annotation])
	# if the metabolite has a chebi number
	if 'CHEBI' in data['database_links']:
		chebi_links = data['database_links']['CHEBI']
	# check if any of the MNX entries correspond to ChEBI IDs
	for mnx_id in data['database_links'].get('MetaNetX (MNX) Chemical', set()):
		# search MNX for ChEBI IDs
		try:
			mnx_response = requests.get('https://rdf.metanetx.org/chem/'+mnx_id)
			# if the request wasn't successful
			if mnx_response.status_code != 200:
				print(f'MNX request failed for {mnx_id} (for met {met.id})')
				continue
			try:
				for i in [x for x in json.loads(BeautifulSoup(mnx_response.text, 'xml').body.script.string)['sameAs'] if 'chebi' in x.lower()]:
					chebi_num = i.split('/')[-1].replace('CHEBI:','')
					chebi_links.add('CHEBI:'+chebi_num)
			except (json.JSONDecodeError, KeyError, TypeError, AttributeError) as e:
				warn(f'Error parsing MNX response for {mnx_id} (for met {met.id}): {e}. May have to rerun later.',ignore_duplicate_warnings=True)
				continue
		except (ConnectionError, OSError, requests.exceptions.RequestException) as e:
			warn(e,ignore_duplicate_warnings=True)
	# search metabolite's rxns for KEGG IDs or EC numbers to find KEGG Compound IDs
	if 'kegg.compound' not in met.annotation:
		for rxn in met.reactions:
			# check if ec-code in rxn.annotation
			if 'ec-code' in rxn.annotation:
				ec_codes = set(i.strip() for i in rxn.annotation['ec-code'].split(','))
				for ec_code in ec_codes:
					# search EC for ChEBI IDs
					ec_response = requests.get('http://rest.kegg.jp/get/'+ec_code)
					# if the request wasn't successful
					if ec_response.status_code != 200:
						print(f'EC request failed for {ec_code} (for met {met.id})')
						continue
					ec_data = ec_response.text
					# find "SUBSTRATE " and "PRODUCT " sections (each may have multiple lines); extract KEGG Compound IDs from these sections
					substrate_section = re.search(r'SUBSTRATE\s+(.+?)(\n\S|\Z)', ec_data, re.DOTALL)
					product_section = re.search(r'PRODUCT\s+(.+?)(\n\S|\Z)', ec_data, re.DOTALL)
					# extract KEGG Compound IDs from substrate section (everything after "CPD:")
					kegg_stoich = {}
					if substrate_section:
						substrates = re.findall(r'\[CPD:(\S+)\]', substrate_section.group(1))
						for sub in substrates:
							kegg_stoich[sub] = kegg_stoich.get(sub,0) - 1
					# extract KEGG Compound IDs from product section (everything after "CPD:")
					if product_section:
						products = re.findall(r'\[CPD:(\S+)\]', product_section.group(1))
						for prod in products:
							kegg_stoich[prod] = kegg_stoich.get(prod,0) + 1
					
					side_in_model = rxn.reactants if met in rxn.reactants else rxn.products
					# Identify correct side of rxn based on length of substrates/products lists (e.g., to avoid isomers from isomerase reactions)
					kegg_ids_in_model_side = set(i.annotation['kegg.compound'] for i in side_in_model if 'kegg.compound' in i.annotation)
					for side in (substrates, products):
						# if only 1 substrate/product differs between model and KEGG, assume it's the current metabolite
						if len(side) == len(side_in_model):
							unknown_kegg_ids = set(side).difference(kegg_ids_in_model_side)
							if len(unknown_kegg_ids) == 1:
								# if total kegg_stoich on this side matches the model's stoichiometry, add the unknown kegg_id
								total_stoich = sum([kegg_stoich[i] for i in side if i in kegg_stoich])
								model_stoich = sum([rxn.get_coefficient(i.id) for i in side_in_model])
								# don't check for exact stoichiometry match, since EC number pages generally lack them
								data['database_links'].setdefault('KEGG Compound', set()).add(unknown_kegg_ids.pop())
						
			if 'kegg.reaction' in rxn.annotation:
				kegg_rxn_response = requests.get('http://rest.kegg.jp/get/'+rxn.annotation['kegg.reaction'])
				if kegg_rxn_response.status_code != 200:
					print(f'KEGG reaction request failed for {rxn.annotation["kegg.reaction"]} (for met {met.id})')
					continue
				kegg_rxn_data = kegg_rxn_response.text
				# find "EQUATION" section and extract KEGG Compound IDs from there
				equation_section = re.search(r'EQUATION\s+(.+?)(\n\S|\Z)', kegg_rxn_data, re.DOTALL)
				if not equation_section:
					continue
				# check which side of the equation the metabolite is on
				substrate_section, product_section = equation_section.group(1).split(' <=> ') if ' <=> ' in equation_section.group(1) else equation_section.group(1).split(' => ')
				kegg_stoich = {}
				substrates = re.findall(r'\b(C\d+)\b', substrate_section)
				for sub in substrates:
					kegg_stoich[sub] = kegg_stoich.get(sub,0) - 1
				products = re.findall(r'\b(C\d+)\b', product_section)
				for prod in products:
					kegg_stoich[prod] = kegg_stoich.get(prod,0) + 1
				side_in_model = rxn.reactants if met in rxn.reactants else rxn.products
				# Identify correct side of rxn based on length of substrates/products lists (e.g., to avoid isomers from isomerase reactions)
				kegg_ids_in_model_side = set(i.annotation['kegg.compound'] for i in side_in_model if 'kegg.compound' in i.annotation)
				for side in (substrates, products):
					# if only 1 substrate/product differs between model and KEGG, assume it's the current metabolite
					if len(side) == len(side_in_model):
						unknown_kegg_ids = set(side).difference(kegg_ids_in_model_side)
						if len(unknown_kegg_ids) == 1:
							# if total kegg_stoich on this side matches the model's stoichiometry, add the unknown kegg_id
							total_stoich = sum([kegg_stoich[i] for i in side if i in kegg_stoich])
							model_stoich = sum([rxn.get_coefficient(i.id) for i in side_in_model])
							if abs(total_stoich) == abs(model_stoich):
								data['database_links'].setdefault('KEGG Compound', set()).add(unknown_kegg_ids.pop())
							else:
								print(f'Stoichiometry mismatch when checking KEGG reaction {met.annotation["kegg.reaction"]} for met {met.id}')
	for kegg_id in data['database_links'].get('KEGG Compound', set()):
		# search KEGG for ChEBI IDs
		kegg_response = requests.get('http://rest.kegg.jp/get/'+kegg_id)
		# if the request wasn't successful
		if kegg_response.status_code != 200:
			print(f'KEGG request failed for {kegg_id} (for met {met.id})')
			continue
		kegg_data = kegg_response.text
		# find the DBLINKS section
		dblinks_section = re.search(r'DBLINKS\s+(.+?)(\n\S|\Z)', kegg_data, re.DOTALL)
		if dblinks_section:
			dblinks_text = dblinks_section.group(1)
			# find all ChEBI IDs in the DBLINKS section
			chebi_ids = re.findall(r'ChEBI:\s*(\d+)', dblinks_text)
			for chebi_id in chebi_ids:
				chebi_links.add('CHEBI:'+chebi_id)
	for pubchem_id in data['database_links'].get('PubChem Compound', set()):
		# search PubChem for ChEBI IDs
		pubchem_response = requests.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{pubchem_id}/JSON')
		# if the request wasn't successful
		if pubchem_response.status_code != 200:
			print(f'PubChem request failed for {pubchem_id} (for met {met.id})')
			continue
		pubchem_data = pubchem_response.json()
		# find all ChEBI IDs in the response
		for item in pubchem_data.get('Record', {}).get('Reference', []):
			if 'https://www.ebi.ac.uk/chebi/searchId' in item.get('URL',''):
				# extract the ChEBI ID from the URL (everything after the 'chebiId=')
				chebi_id = item.get('URL','').split('?chebiId=')[-1]
				if chebi_id:
					if 'CHEBI:' not in chebi_id:
						print(f'Extracted ChEBI ID {chebi_id} from PubChem URL for met {met.id}')
					else:
						chebi_links.add(chebi_id)
	if chebi_links:
		ids_found.add('chebi')
		# remove all duplicates
		chebi_ids_for_met = set()
		for chebi_id in chebi_links:
			if chebi_id in chebi_ids_for_met:
				continue
			chebi_ids_for_met.add(chebi_id)
			num = chebi_id.replace('CHEBI:','')
			if num == '':
				continue
			# search ChEBI for the number
			chebi_response = requests.get('https://www.ebi.ac.uk/chebi/backend/api/public/compound/'+num)
			# if the request wasn't successful
			if chebi_response.status_code != 200:
				print(f'ChEBI request failed for {num} (for met {met.id})')
				continue
			# get the data in json format
			chebi_data = chebi_response.json()
			# get the xml data
			# chebi_data = BeautifulSoup(chebi_response.text, 'xml')
			# print(chebi_data)
			# find the chebiId tag
			# id_tag = chebi_data.find('chebiId')
			id_tag = chebi_data.get('id', None)
			chemical_data = chebi_data.get('chemical_data', {})
			perfect_match = False
			potential_matches_to_review.append(chebi_id)
			# if the tag was found, add it to the map and model
			if not chemical_data:
				print(f'No chemical data found for CHEBI ID {num} (for met {met.id})')
			elif not id_tag:
				print(f'No CHEBI ID found in data for CHEBI ID {num} (for met {met.id})')
			else:
				# compare formulas and charges to ensure match
				perfect_match = compare_metabolite_to_database(met,
															[chemical_data.get('formula', '')],
															[chemical_data.get('charge', None)])
				if not perfect_match:
					continue
			# check each entry in chebi_data["database_accessions"]["MANUAL_X_REF"] (if present) to see if any have "prefix" == "kegg.compound" that match met.annotation['kegg.compound']
			if 'kegg.compound' in met.annotation:
				manual_xrefs = chebi_data.get('database_accessions', {}).get('MANUAL_X_REF', [])
				manual_xref_ids = [xref.get("id","") for xref in manual_xrefs if xref.get("prefix","").lower() == "kegg.compound"]
				kegg_compound_id = met.annotation['kegg.compound']
				kegg_match_found = False
				for xref in manual_xrefs:
					if xref.get('prefix','').lower() == 'kegg.compound' and (xref.get('id','') == kegg_compound_id or xref.get('accession_number','') == kegg_compound_id):
						kegg_match_found = True
						break
				if not kegg_match_found and len(manual_xref_ids) > 0:
					warn(f'KEGG Compound mismatch for {met.id}: model has {kegg_compound_id}, ChEBI {id_tag} has {manual_xref_ids}',ignore_duplicate_warnings=True)
					# print(f'\t{chebi_links}')
					continue
			match_found = True
			num_primary = str(id_tag)
			met_chebi_map.setdefault(num_primary,{'default':base_id})[met.compartment] = met.id
			# add the annotation to the model
			met.annotation['chebi'] = 'CHEBI:'+num_primary
			# print('Found ChEBI ID',num_primary,'for',base_id)
			if perfect_match:
				break
		# met_chebi_map.setdefault(data['chebi'].replace('CHEBI:',''),{'default':base_id})[met.compartment] = met.id
	if not match_found:
		print(f'ChEBI ID not found for metabolite {met.id} (base ID {base_id}). Potential matches: {set(potential_matches_to_review)}')
		chebi_ids_ambiguous.update(set(i.replace('CHEBI:','') for i in potential_matches_to_review)) # store ambiguous chebi IDs for review, so ontology can be checked later (to help identify cofactors that may have been missed)
save_cobra_model(model,SM_path)

ChEBI ID not found for metabolite trnaarg_c (base ID trnaarg). Potential matches: {'CHEBI:29171', 'CHEBI:10673', 'CHEBI:15167'}
EC request failed for 1.1.1.n12 (for met 3hdd5coa_x)
ChEBI ID not found for metabolite 3hdd5coa_x (base ID 3hdd5coa). Potential matches: {'CHEBI:15456'}
ChEBI ID not found for metabolite trnaphe_m (base ID trnaphe). Potential matches: {'CHEBI:10687', 'CHEBI:15183', 'CHEBI:29184'}
ChEBI ID not found for metabolite but2eACP_m (base ID but2eACP). Potential matches: set()
ChEBI ID not found for metabolite trdox_n (base ID trdox). Potential matches: {'CHEBI:18191', 'CHEBI:15240', 'CHEBI:7845', 'CHEBI:14726'}
ChEBI ID not found for metabolite 2dr1p_c (base ID 2dr1p). Potential matches: {'CHEBI:48460', 'CHEBI:58576', 'CHEBI:57259', 'CHEBI:11563', 'CHEBI:1081', 'CHEBI:19558', 'CHEBI:28542'}
No chemical data found for CHEBI ID 5025 (for met ficytb5_r)
No chemical data found for CHEBI ID 18097 (for met ficytb5_r)
No chemical data found for CHEBI ID 14243 (for met ficytb

In [16]:
# with open('~/Downloads/bigg_mets.json', 'r') as f:
with open('./input/met_chebi_map.json', 'r') as f:
	file = json.load(f)
	if not met_chebi_map: 
		met_chebi_map = file

with open('./input/met_chebi_map.json','w') as f:
	json.dump(file,f,indent=4)

In [17]:
# add some default mappings in case they are not found in the model
# met_chebi_map_defaults = {'24875':{'default':'feCation'},'60240':{'default':'metal2'}}
met_chebi_map_defaults = {}
# add the default mappings to the main dictionary if they are not already there
for key in met_chebi_map_defaults.copy():
	if key not in met_chebi_map:
		met_chebi_map[key] = met_chebi_map_defaults[key]
# save the dictionary to a file
with open('./input/met_chebi_map.json','w') as f:
	json.dump(met_chebi_map,f,indent=4)

# download proteome files

In [18]:
import gzip
import shutil
# get proteome data from uniprot, for creating a model
def get_uniprot_proteome_data(url):
    with requests.get(url, stream=True) as request:
        request.raise_for_status()
        with open('./input/stream.gz', 'wb') as f:
            for chunk in request.iter_content(chunk_size=2**20):
                f.write(chunk)
    # uncompress the file
    return gzip.open('./input/stream.gz', 'rb')

f_in = get_uniprot_proteome_data(url)

# uncompress file
with open('./input/proteome.tsv', 'wb') as f_out:
	shutil.copyfileobj(f_in, f_out)
# read the file into a pandas DataFrame
df = pd.read_csv('./input/proteome.tsv', sep='\t')

# sort the DataFrame by a specific column, e.g., 'Entry'
df = df.sort_values(by='Entry')

# if there are extra accessions, search only the ones not already in the DataFrame
if extra_accessions:
	# get the list of accessions already in the DataFrame
	existing_accessions = df['Entry'].tolist()
	# get the list of accessions to search for
	accessions_to_search = [acc for acc in extra_accessions if acc not in existing_accessions]
	# search for the accessions using build_uniprot_query and get_uniprot_proteome_data
	f_in = get_uniprot_proteome_data(build_uniprot_query(uniprot_ids=accessions_to_search))
	# read the new file directly from the gzip stream
	new_df = pd.read_csv(f_in, sep='\t')
	# append the new DataFrame to the existing one
	df = pd.concat([df, new_df], ignore_index=True)

# save the sorted DataFrame back to the file
df.to_csv('./input/proteome.tsv', sep='\t', index=False)

HTTPError: 400 Client Error:  for url: https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession,reviewed,id,protein_name,gene_names,organism_name,annotation_score,ec,protein_families,sequence,mass,cc_sequence_caution,ft_chain,cc_ptm,cc_cofactor,cc_subunit,encoded_in,cc_subcellular_location,go_c,ft_intramem,ft_transmem,go_id,keyword,cc_function,cc_activity_regulation,ph_dependence,cc_miscellaneous,cc_alternative_products&format=tsv&query=%28accession:A0A2U9RAK4%20OR%20accession:A0A2U9RAV5%20OR%20accession:A0A2U9RBW3%20OR%20accession:A0A2U9RBK5%20OR%20accession:A0A2U9RCF9%20OR%20accession:A0A2U9RAR9%20OR%20accession:A0A2U9RBJ3%20OR%20accession:A0A2U9RCC6%20OR%20accession:A0A2U9RB22%20OR%20accession:A0A2U9RCH4%20OR%20accession:A0A2U9RAU9%20OR%20accession:A0A2U9RAN3%20OR%20accession:A0A2U9RB35%20OR%20accession:A0A2U9RAL6%20OR%20accession:A0A099NYH5%20OR%20accession:A0A099P8Y8%20OR%20accession:A0A099P8Z3%20OR%20accession:A0A099NV60%20OR%20accession:A0A099P492%20OR%20access

# draft protein stoichiometry sheet

In [18]:
compartments_lowercase = {key: [value.lower()] for key, value in model.compartments.items()}
cytosol_id = RBA_defaults.cytosol # default cytosol compartment ID
display(compartments_lowercase)
# map compartments to their specific membrane compartments, for better matching of localization terms and enzymes
compartment_adjacency_map = {}
# putting 1 term into multiple compartments ensures that all matches to that term will be found (e.g., 'endomembrane system' is ambiguous and could represent multiple compartments, but whether to include it arguably depends on which other terms are present since it may just be part of a hierarchical series of terms)
# add 'cytoplasm, cytosol' to whichever compartment contains 'cytoplasm' or 'cytosol'
for key, values in compartments_lowercase.items():
	val = values[0]  # get the first value in the list
	# update in a way that's flexible to different model compartment names
	# cell septum: it could arguably be en or e depending on where in the cell wall it's facing, but the only protein with it is uncharacterized, so include both just to err on the side of being lenient
	if val in ['cytoplasm', 'cytosol']: 
		compartments_lowercase[key] += [
								  'cytosol','cytoplasm, cytosol','cytoplasm, cytoskeleton, actin patch','cytoplasm, cytoskeleton, microtubule organizing center, spindle pole body','cytoplasm, cytoskeleton, microtubule organizing center','cytoplasm, cytoskeleton, spindle pole','cytoplasm, cytoskeleton, spindle','cytoplasm, cytoskeleton','cytoplasm, perinuclear region','bud neck',
								  'note=predominantly cytoplasmic','note=nucleus and/or cytoplasm','note=shuttles between cytoplasm and nucleus/nucleolus','note=localizes to the plus ends of microtubules and the mitotic spindle poles','note=shuttles between the nucleus and the cytoplasm',
								  'note=cytoplasmic face of coated pits and vesicles','cytoplasmic side of endoplasmic reticulum membrane','cytoplasmic side of mitochondrial outer membrane'
								  ]
		cytosol_id = key  # store the ID of the cytosol compartment
	elif val in ['cell envelope']:
		compartments_lowercase[key] += [
								  'cell membrane','cell surface',
								  'periplasm','periplasmic space','cell septum']
	elif val in ['endoplasmic reticulum']:
		compartments_lowercase[key] += [
			'endoplasmic reticulum lumen', 'concentrated in punctae along the er tubules', 'microsome',
			'note=enriched in the cortical er','cortical endoplasmic reticulum lumen','perinuclear endoplasmic reticulum lumen'
			]
	elif val in ['endoplasmic reticulum membrane']:
		compartments_lowercase[key] += [
								  'microsome membrane','endoplasmic reticulum-golgi intermediate compartment membrane']
	elif val in ['extracellular', 'extracellular space']:
		compartments_lowercase[key] += ['secreted','secreted, cell wall','extracellular matrix','extracellular region',
								  'cell projection','cell septum']
	elif val in ['golgi']:
		compartments_lowercase[key] += [
								  'golgi lumen','golgi apparatus', 'golgi apparatus, golgi stack','golgi apparatus, cis-golgi network','golgi apparatus, trans-golgi network']
	elif val in ['golgi membrane']:
		compartments_lowercase[key] += [
								  'golgi apparatus membrane','golgi apparatus, cis-golgi network membrane','golgi apparatus, trans-golgi network membrane',
								  'endoplasmic reticulum-golgi intermediate compartment membrane']
	elif val in ['lipid particle']:
		compartments_lowercase[key] += ['lipid droplet']
	elif val in ['mitochondrion']:
		compartments_lowercase[key] += ['note=predominantly mitochondrial','mitochondrial matrix','mitochondrion matrix','mitochondrion matrix, mitochondrion nucleoid','mitochondrial nucleoid','mitochondrial chromosome','note=predominantly in the mitochondria and in the nucleus',
								  'matrix side of mitochondrial inner membrane',
								  'mitochondrial ribosome','mitochondrial large ribosomal subunit','mitochondrial small ribosomal subunit']
	elif val in ['mitochondrial membrane']:
		compartments_lowercase[key] += ['mitochondrion membrane','mitochondrion inner membrane','mitochondrion outer membrane','mitochondrion intermembrane space','mitochondrial intermembrane space','[phosphatidylserine decarboxylase 1 alpha chain]: mitochondrion inner membrane',
								  'mitochondrial inner-outer membrane contact site']
	elif val in ['nucleus']:
		compartments_lowercase[key] += ['note=nucleus and/or cytoplasm','note=shuttles between cytoplasm and nucleus/nucleolus','note=shuttles between the nucleus and the cytoplasm','nuclear chromosome',
								  'chromatin','nucleolus','nucleolar peripheral inclusion body','heterochromatin island','euchromatin','nuclear speck','nuclear body','nuclear microtubule','nuclear periphery',
								  'nuclear matrix','nucleoplasm','note=resides mostly in the nucleoli and relocalizes to the nucleoplasm upon dna damage',
								  'note=predominantly in the mitochondria and in the nucleus','nucleus envelope','nucleus inner membrane','nucleus matrix','nucleus membrane','nucleus, nuclear pore complex','nucleus, nucleolus','nucleus, nucleoplasm', 'chromosome, centromere', 'chromosome, centromere, kinetochore', 'chromosome', 'chromosome, telomere','chromosome, telomeric region','condensed nuclear chromosome']
	elif val in ['peroxisome']:
		compartments_lowercase[key] += ['peroxisome matrix','peroxisomal matrix',
								  'peroxisome membrane','peroxisomal membrane']
	elif val in ['vacuole']:
		compartments_lowercase[key] += ['vacuole lumen','vacuolar lumen']
	elif val in ['vacuolar membrane']:
		compartments_lowercase[key] += ['prevacuolar compartment membrane','note=during pexophagy, accumulates in the vacuolar membrane region, where the peroxisomes contact the vacuole',
								  'vacuole membrane']
# membrane-bound compartments to add to cytoplasm if not already present (since they're not always separate compartments in models)
extra_membrane_terms = ['lysosome', 'endosome', 'endosome membrane', 'endosome, multivesicular body membrane', 'preautophagosomal structure membrane','preautophagosomal structure', 'cytoplasm, p-body','cytoplasmic vesicle membrane','cytoplasmic vesicle, autophagosome membrane','cytoplasmic vesicle, copi-coated vesicle membrane','cytoplasmic vesicle, copii-coated vesicle membrane','cytoplasmic vesicle, secretory vesicle membrane','cytoplasmic vesicle, secretory vesicle']
for term in extra_membrane_terms:
	if term not in [value for values in compartments_lowercase.values() for value in values]:
		# if the term is not already present, add it to the cytoplasm compartment
		compartments_lowercase[cytosol_id].append(term)
compartments_lowercase_values = [value for values in compartments_lowercase.values() for value in values]
# expressions that don't represent specific compartments, to ignore when assigning compartments
location_expressions_to_ignore = ['membrane, coated pit','note=the ermes/mdm complex localizes to a few discrete foci (around 10 per single cell), that represent mitochondria-endoplasmic reticulum junctions','these foci are often found next to mtdna nucleoids']

# mapped based on adjacency in the cell, starting from extracellular space going inward. Multiple terms used to account for different model formats and annotations.
compartment_adjacency_map_terms = {
	('extracellular', 'extracellular space'): {
		('cell envelope',): {
			('cytoplasm', 'cytosol'): {
				('endoplasmic reticulum membrane',): {
					('endoplasmic reticulum',): {}
				},
				('golgi membrane',): {
					('golgi',): {}
				},
				('vacuolar membrane', 'vacuole membrane'): {
					('vacuole',): {}
				},
				('mitochondrial membrane', 'mitochondrion membrane'): {
					('mitochondrion', 'mitochondria'): {}
				},
				('lipid particle',): {},
				('peroxisomal membrane', 'peroxisome membrane'): {
					('peroxisome',): {}
				},
				('nuclear membrane', 'nucleus membrane'): {
					('nucleus',): {}
				},
			}
		}
	}
}
# create compartment_adjacency_map based on compartment_adjacency_map_terms and compartments_lowercase
def add_to_adjacency_map(adj_map, terms, compartments_lowercase, parent_compartment_ids=[]):
	"""Recursively add compartments to the adjacency map based on the terms and compartments_lowercase."""
	for term_list, subterms in terms.items():
		# find compartment IDs for the term_list
		compartment_ids = []
		for term in term_list:
			for comp_id, comp_terms in compartments_lowercase.items():
				if term in comp_terms:
					compartment_ids.append(comp_id)
		# add to adjacency map
		for comp_id in compartment_ids:
			if comp_id not in adj_map:
				adj_map[comp_id] = set()
			# add parent compartments as adjacent compartments
			for parent_comp_id in parent_compartment_ids:
				# adj_map[comp_id].add(parent_comp_id)
				if parent_comp_id not in adj_map:
					adj_map[parent_comp_id] = set()
				if comp_id != parent_comp_id:
					adj_map[parent_comp_id].add(comp_id)
			# recursively add subterms
			# add_to_adjacency_map(adj_map, subterms, compartments_lowercase, parent_compartment_ids + [comp_id])
			add_to_adjacency_map(adj_map, subterms, compartments_lowercase, [comp_id])
	return adj_map
compartment_adjacency_map = add_to_adjacency_map({}, compartment_adjacency_map_terms, compartments_lowercase)
# convert sets to lists for JSON serialization
for key in compartment_adjacency_map:
	compartment_adjacency_map[key] = sorted(list(compartment_adjacency_map[key]))
# warn if any compartments aren't in the adjacency map
for comp_id in compartments_lowercase.keys():
	if comp_id not in compartment_adjacency_map:
		warn(f"Compartment '{comp_id}' not found in compartment_adjacency_map. Mapping of enzymes to proteins in other compartments may be incomplete. See compartment_adjacency_map_terms for example of how to add compartments.")
# save compartment_adjacency_map to file
with open('./input/compartment_adjacency_map.json','w') as f:
	json.dump(compartment_adjacency_map,f,indent=4)

{'e': ['extracellular'],
 'c': ['cytoplasm'],
 'm': ['mitochondrion'],
 'n': ['nucleus'],
 'l': ['lipid particle'],
 'g': ['golgi'],
 'x': ['peroxisome'],
 'v': ['vacuole'],
 'r': ['endoplasmic reticulum'],
 'mm': ['mitochondrial membrane'],
 'rm': ['endoplasmic reticulum membrane'],
 'en': ['cell envelope'],
 'gm': ['golgi membrane'],
 'vm': ['vacuolar membrane']}

In [ ]:
from itertools import product
locations_unmapped = set()
def location_to_compartment(location,include_unmapped=False):
	"""Convert a location string to a compartment string.
	
	**location**: str with the location of the protein, e.g. 'cytoplasm', 'mitochondria', etc.
	**include_unmapped**: bool indicating whether to include unmapped locations in the output (False for quicker processing, True for debugging and easier curation).
	
	Output: str with the compartment name, e.g. 'c', 'm', etc.
	"""
	if pd.isna(location):
		return ''
	location = location.lower()
	# check if the location(s) are in the values of compartments_lowercase
	if location in compartments_lowercase_values:
		# return all compartments (as a list) that match the location
		return [k for k, v in compartments_lowercase.items() if location in v]
	elif location in location_expressions_to_ignore:
		return []
	# compartment = next((k for k, v in compartments_lowercase.items() if location in v), location)
	# if the compartment is not found, return the original location
	else:
		compartment = [location] if include_unmapped else []
		# print(f"Warning: {location} not found in compartments_lowercase, using original location")
		locations_unmapped.add(location)
	return compartment

# create draft of protein stoich sheet from the file
df = pd.read_csv('./input/proteome.tsv', sep='\t')
df['uniprot'] = df['Entry']
# get the protein ID from the gene_protein_mapping key with a value that contains the uniprot ID
for i in df.index:
	gene_src = [k for k, v in gene_protein_mapping.items() if df.at[i,'uniprot'] in v]
	df.at[i,'gene_src'] = gene_src[0] if gene_src else ''
	# sort the list by name and then by whether they have "notes" in their attributes (since in iIsor850, this is where lower e-value BLAST hits are denoted)
	if len(gene_src) > 1:
		df.at[i,'gene_src'] = sorted(gene_src,key=lambda x: ('sc_bidirectional_blast' in model.genes.get_by_id(x).__dict__['notes'].keys(), x))[0]

# if any of the keys in gene_protein_mapping don't have a protein with them as their gene_src, print a warning
def get_protein_id_from_gene(gene_id,protein_ids):
	if protein_ids[0] is not None and gene_id not in df['gene_src'].values:
		for protein_id in protein_ids:
			if protein_id in df['uniprot'].values:
				print(f"Warning: {gene_id} has a protein ID in the proteome file ({protein_ids}) but its gene_src is {df.loc[df['uniprot'] == protein_id, 'gene_src'].values[0]}")
				return
		print(f"Warning: {gene_id} does not have a protein ID in the proteome file")
for gene_id, protein_ids in gene_protein_mapping.items():
	get_protein_id_from_gene(gene_id, protein_ids)

df['protein'] = df['Protein names']
# move 'Sequence' to 'sequence', and delete the original 'Sequence' column
df['sequence'] = df['Sequence']
df.drop(columns=['Sequence'], inplace=True)
# check "Subunit structure" for the lowest word ending in "mer"
def get_mer(prot,excluded_prefixes=[],required_prefixes=[],adjust_for_cleavage=True,find_for_whole_complex=False,column_to_check='Subunit structure',str_to_check=None):
	"""Get the stoichiometry of a protein from its subunit structure description.
	**prot**: DataFrame row with the protein information.
	**excluded_prefixes**: list of str prefixes to not allow in front of considered words (e.g., 'hetero' to exclude heterooligomers).
	**required_prefixes**: list of str prefixes that must be present in front of considered words (e.g., 'homo' to only include homooligomers).
	**adjust_for_cleavage**: Bool indicating whether to adjust the stoichiometry based on cleavage information in the DataFrame row.
	**find_for_whole_complex**: Bool indicating whether to find the stoichiometry for the whole complex or just the chosen protein. Useful for reducing unnecessary error reports for specific enzymes.
	"""
	global df
	protein_copies_per_complex = None
	if column_to_check:
		x = str(prot[column_to_check]).lower()
	else:
		x = str_to_check.lower() if str_to_check is not None else ""
	# default terms that otherwise are inaccurately processed 
	if x in ("subunit: component of the er-mitochondria encounter structure (ermes) or mdm complex, composed of mmm1, mdm10, mdm12 and mdm34. a mmm1 homodimer associates with one molecule of mdm12 on each side in a pairwise head-to-tail manner, and the smp-ltd domains of mmm1 and mdm12 generate a continuous hydrophobic tunnel for phospholipid trafficking. {eco:0000256|hamap-rule:mf_03104}."):
		protein_copies_per_complex = 1
	elif x.startswith("subunit: interacts with the flavoprotein subunit within the sdh catalytic dimer."): # for Succinate dehydrogenase
		protein_copies_per_complex = 1
	elif x.startswith("subunit: component of the ubiquinol-cytochrome c oxidoreductase (cytochrome b-c1 complex, complex iii, ciii), a multisubunit enzyme composed of 3 respiratory subunits cytochrome b, cytochrome c1 and rieske protein, 2 core protein subunits, and additional low-molecular weight protein subunits. the complex exists as an obligatory dimer and forms supercomplexes (scs) in the inner mitochondrial membrane with cytochrome c oxidase (complex iv, civ)."): # for Cytochrome b-c1 complex
		protein_copies_per_complex = 1
	elif x.startswith("subunit: f-type atpases have 2 components, cf(1) - the catalytic core - and cf(0) - the membrane proton channel. in yeast, the dimeric form of atp synthase consists of 17 polypeptides: alpha, beta, gamma, delta, epsilon, 4 (b), 5 (oscp), 6 (a), 8, 9 (c), d, e (tim11), f, g, h, i/j and k."):
		if find_for_whole_complex:
			protein_copies_per_complex = 17
		else:
			protein_copies_per_complex = 1
	elif pd.notna(prot['Gene Ontology (cellular component)']) and "fatty acid synthase complex [GO:0005835]" in prot['Gene Ontology (cellular component)']:
		if find_for_whole_complex:
			protein_copies_per_complex = 12
		else:
			protein_copies_per_complex = 6
	elif x.startswith("subunit: dimer of alpha and beta chains."): # restricted since Tubulin alpha chain and Tubulin beta chain are incorrectly labeled as a heterotetramer, not a heterodimer
		protein_copies_per_complex = 1
	elif x.startswith("subunit: homohexamer. dimer of trimers."):
		protein_copies_per_complex = 6
	elif not find_for_whole_complex and x.startswith("subunit: octamer of two non-identical subunits"):
		protein_copies_per_complex = 4
	elif not find_for_whole_complex and x.startswith("subunit: tetramer of two alpha and two beta subunits."):
		protein_copies_per_complex = 2
	elif not find_for_whole_complex and re.match(r"subunit: v-atpase is a heteromultimeric enzyme composed of a peripheral catalytic v1 complex \(components a to h\) attached to an integral membrane v0 proton pore complex \(components: a, c, c', c'', d, e, f and voa1\).*. the decameric c-ring forms the proton-conducting pore, and is composed of eight proteolipid subunits c, one subunit c' and one subunit c''", x):
			# do only when find_for_whole_complex is False, to avoid implying the whole complex uses 8 subunits when aligning model and UniProt data
			protein_copies_per_complex = 1
			# make exception (set stoich to 8) if its associated gene in the model is vma3, which encodes subunit c
			if 'gene_src' in prot and prot['gene_src'] in model.genes:
				gene = model.genes.get_by_id(prot['gene_src'])
				if 'vma3' in str(gene.name).lower():
					protein_copies_per_complex = 8
	# default processing
	else:
		try:
			# find the lowest value (monomer, dimer, trimer, etc.)
			words = ['monomer','dimer','trimer','tetramer','pentamer','hexamer','heptamer','octamer','nonamer','decamer']
			for word,i in zip(words,range(1,len(words)+1)):
				# if the word is found, return the value
				if word in x:
					# check if there are any matches without excluded prefixes
					prefix_excluded = False
					for prefix in excluded_prefixes:
						# if no matches for word without the prefix, exclude this word
						if re.search(r'\b'+prefix+word, x):
							prefix_excluded = True
							for match in re.finditer(word, x):
								# check if the match is preceded by the excluded prefix
								if not re.search(r'\b'+prefix+word, x[max(0,match.start()-len(prefix)-1):match.end()]):
									prefix_excluded = False
									break
					if prefix_excluded:
						continue
					# check if required prefixes are present
					if required_prefixes:
						if not any(re.search(r'\b'+prefix+word, x) for prefix in required_prefixes):
							continue
					if (protein_copies_per_complex is None) or i < protein_copies_per_complex:
						protein_copies_per_complex = i
					# check for phrases indicating only that a certain complex is active; if so, override the stoichiometry
					elif re.search(r'only the \w*'+word+r'\b is catalytically active', x):
						protein_copies_per_complex = i
						break
			else:
				# If no other stoich. found, set homooligomers to 2 if "homooligomer" is mentioned (since it implies >1 subunit)
				if protein_copies_per_complex is None and 'homooligomer' in x and 'homo' not in excluded_prefixes:
					protein_copies_per_complex = 2
		except:
			pass
	protein_copies_per_complex = 1 if protein_copies_per_complex is None else protein_copies_per_complex
	# check "Protein names" for "Cleaved into: " and parse the cleavage stoichiometry based on the number of semicolons + 1 (number of products)
	if adjust_for_cleavage:
		cleavage = re.search(r'\[Cleaved into:(.*?)\]', prot['Protein names'])
		if cleavage:
			num_products = cleavage.group(1).count(';') + 1
			# update subunits/copy column in df
			df.loc[df['Entry'] == prot['Entry'], 'subunits/copy'] = num_products
			if num_products > 1:
				# divide the stoichiometry by the number of products (rounding up to nearest integer)
				protein_copies_per_complex = -(-protein_copies_per_complex // num_products)
				print(f"{prot['Entry']} cleaved into {num_products} products, adjusting stoich. to {protein_copies_per_complex}")
		elif 'hetero' not in excluded_prefixes and ('hetero' not in required_prefixes):
			# exclude 'hetero' if cleavage isn't accounted for, since heteromer stoich. is only used for special cases
			protein_copies_per_complex = get_mer(prot,excluded_prefixes=excluded_prefixes + ['hetero'], required_prefixes=required_prefixes, adjust_for_cleavage=False)
	return protein_copies_per_complex # if no value is found, assume monomer

def chebi_in_model(chebi_id):
	global chebi_ids_unknown
	if str(chebi_id) in met_chebi_map:
		return True
	if chebi_id not in chebi_ids_searched:
		print(f"CHEBI ID {chebi_id} not found in met_chebi_map when updating protein stoichiometry")
	chebi_ids_unknown.add(chebi_id)
	return False

def add_metabolite_to_met_chebi_map(cof_dict,cof,chebi_id_2):
	global met_chebi_map
	chebi_id_2 = str(chebi_id_2)
	cof_dict[chebi_id_2] = deepcopy(cof_dict[cof])
	cof_dict[chebi_id_2]['chebi'] = chebi_id_2
	mcm_2 = deepcopy(met_chebi_map[chebi_id_2])
	# add to met_chebi_map if it doesn't exist yet
	# met_chebi_map.setdefault(str(cof_dict[cof]['chebi']),mcm_2)
	# merge with met_chebi_map if it's not already in cofactor dict, to avoid redundancy where possible (e.g., if a metabolite has different IDs in different compartments)
	for cof2 in cof_dict:
		if str(cof_dict[cof2]['chebi']) == chebi_id_2:
			break
	# else:
	# 	met_chebi_map[str(cof_dict[cof]['chebi'])].update(deepcopy(met_chebi_map[chebi_id_2]))
	return cof_dict

chebi_ids_searched = set()
chebi_data_cache = {}
def search_chebi(id,add_to_cofactors_if_found=True,update_match_found=True,search_children=True):
	global match_found, cof_dict, chebi_children, chebi_ids_searched, chebi_data_cache, cof, chebi_id_2, met_chebi_map
	chebi_data = {}
	if (not id) and '0' not in str(id): # `'0' not in id` in case ChEBI ID consisting only of zeroes is ever used
		return chebi_data
	if add_to_cofactors_if_found and chebi_in_model(id):
		# print(f'CHEBI ID {id} in model; adding to cofactor dict {cof_dict}.')
		if update_match_found:
			match_found = True
		cof_dict = add_metabolite_to_met_chebi_map(cof_dict,cof,id)
	if id in chebi_data_cache:
		chebi_data = chebi_data_cache[id]
	else:
		# if id in chebi_ids_searched:
		# 	return chebi_data
		response = requests.get(f'https://www.ebi.ac.uk/chebi/backend/api/public/compound/{id}', stream=True,timeout=30)
		chebi_ids_searched.add(id)
		chebi_data = response.json()
		chebi_data_cache[id] = chebi_data
	# check if this metabolite has secondary IDs that are in the model
	chebi_secondary_ids = chebi_data.get('secondary_ids',[])
	for chebi_secondary_id in chebi_secondary_ids:
		chebi_id_2 = chebi_secondary_id.replace('CHEBI:','')
		chebi_ids_searched.add(chebi_id_2)
		if chebi_in_model(chebi_id_2):
			if update_match_found:
				match_found = True
			print(f'Secondary CHEBI ID {chebi_id_2} in model.')
			cof_dict = add_metabolite_to_met_chebi_map(cof_dict,cof,chebi_id_2)
	chebi_ontology = chebi_data.get('ontology_relations',{})
	if search_children:
		for child in chebi_ontology.get('incoming_relations',[]):
			child_id = child.get('init_id','')
			if child_id in chebi_children:
				# print(f"CHEBI ID {child_id} is child of ID {id}, and already searched; skipping to avoid infinite loop.")
				continue
			if child.get('relation_type','') != 'is a':
				continue
			chebi_children.add(child_id)
			_ = search_chebi(child_id,update_match_found=False,search_children=search_children)
	return chebi_data

cofactor_modifications = []
def add_metabolite_to_protein_stoich(pro, stoich_dict=None, df=df, met_ids_input=None, coeff=1, chebi_id=None, compartment=None):
	"""Add a metabolite to the protein stoichiometry dictionary.
	
	**pro**: Index with the protein information.
	**df**: DataFrame with the protein data.
	**met_ids_input**: set of str with the metabolite IDs to add from the model (if known).
	**coeff**: int with the coefficient to add (positive for products, negative for reactants).
	**chebi_id**: str with the ChEBI ID of the metabolite (optional).
	**compartment**: str with the compartment ID of the metabolite (optional; useful if ChEBI ID maps to multiple compartment-specific metabolites).
	"""
	global chebi_ids_unknown, df_debug, cofactor_modifications
	if met_ids_input is None:
		met_ids = set()
	if chebi_id and not met_ids:
		if chebi_id not in met_chebi_map:
			if chebi_id not in chebi_ids_searched:
				print(f"CHEBI ID {chebi_id} not found in met_chebi_map when updating protein stoichiometry")
			# use the provided protein index 'pro' (not a global 'i')
			try:
				df_debug.loc[pro, ['Entry', 'Cofactor']] = df.loc[pro, ['Entry', 'Cofactor']]
			except Exception:
				# if indexing fails, skip the debug update but still record unknown CHEBI
				pass
			chebi_ids_unknown.add(chebi_id)
			# return the unchanged stoich_dict instead of None to avoid TypeError when callers iterate
			return stoich_dict
		else:
			if not compartment: # get 1st compartment from translation_loc column
				compartment = df.at[pro,'translation_loc'].split(',')[0] if pd.notna(df.at[pro,'translation_loc']) and df.at[pro,'translation_loc'] != '' else None
			# try adding the metabolite from any of the compartments in subloc_assigned column
			for comp in set(df.at[pro,'subloc_assigned'].split(',') + ([compartment] if compartment else [])):
				new_met_id = met_chebi_map.get(chebi_id,{}).get(comp)
				if new_met_id:
					met_ids.add(new_met_id)
					# print(f"Found metabolite ID {new_met_id} for CHEBI ID {chebi_id} in compartment {comp} when updating protein stoichiometry.")
					if comp != compartment:
						cofactor_modifications.append({'pro': df.at[pro,'Entry'], 'met': new_met_id, 'mod':'remove from other compartments'})
		if not met_ids:
			warn(f"Could not find metabolite ID for CHEBI ID {chebi_id} when updating protein stoichiometry",ignore_duplicate_warnings=True)
			try:
				df_debug.loc[pro, ['Entry', 'Cofactor']] = df.loc[pro, ['Entry', 'Cofactor']]
			except Exception:
				pass
			# return the unchanged stoich_dict instead of None to avoid TypeError when callers iterate
			return stoich_dict
	# else:
		# print(f"Adding metabolite IDs {met_ids} to protein {df.at[pro,'Entry']} stoichiometry with coefficient {coeff}.")
	# add met_id to the stoichiometry dictionary
	if stoich_dict is None:
		stoich_dict = [{}]
	final_stoich = []
	# if met_ids:
	# 	print(f"Adding metabolite IDs {met_ids} to protein {df.at[pro,'Entry']} stoichiometry with coefficient {coeff}.")
	for met_id in met_ids:
		stoich_added = deepcopy(stoich_dict)
		if type(stoich_added) is not list:
			stoich_added = [stoich_added]

		if type(stoich_added) is list: # using a list of dicts to represent alternative stoichiometries
			for stoich in stoich_added:
				if met_id in stoich:
					stoich[met_id] = stoich.get(met_id,0) + coeff
				elif met_id != '':
					stoich[met_id] = coeff
		else:
			if met_id != '':
				stoich_added[met_id] = stoich_added.get(met_id,0) + coeff
		final_stoich.extend(stoich_added)
	if not final_stoich and stoich_dict not in (None, [{}]):
		final_stoich = stoich_dict if type(stoich_dict) is list else [stoich_dict]
	return final_stoich

df_debug = pd.DataFrame()
chebi_ids_unknown = set()
for i in df.index:
	df.at[i,'subunits/copy'] = 1
	df.at[i,'stoich'] = get_mer(df.loc[i])
	compartments_encoded = location_to_compartment(df.at[i, 'Encoded in'])
	# if the compartments_encoded is not provided, set to cytosol_id (since it's the most common compartment)
	if (not compartments_encoded) or (compartments_encoded in (None, [], [''])):
		compartments_encoded = [cytosol_id]
	df.at[i, 'translation_loc'] = ','.join(compartments_encoded)

	compartments = compartments_encoded if isinstance(compartments_encoded, list) else [compartments_encoded] # start with the encoded compartments, since if it's translated in them, the protein itself likely exists there even if it's not primarily localized there, which may be important for matching proteomics data
	compartments = compartments_encoded if RBA_defaults.always_localize_in_encoded_compartment else []
	if not pd.isna(df.at[i,'Subcellular location [CC]']):
		# remove 'SUBCELLULAR LOCATION: ' and text between curly brackets for df.at[i,'Subcellular location [CC]'], then split by ';', and convert into comma-separated list of compartments where possible
		# get compartments for each location, flatten the list, remove duplicates, and join as a string
		for loc in re.sub(r'; .*?(\.|$)','. ',re.sub(r'\s+', ' ', re.sub(r' \{.*?\}|\.$', '', df.at[i,'Subcellular location [CC]'].replace('SUBCELLULAR LOCATION: ','')))).strip().split('. '):
			if loc != '':
				cmp = location_to_compartment(loc.split('.')[0].strip())
				if isinstance(cmp, list):
					compartments.extend(cmp)
				else:
					print(f"Warning: {loc} not found in compartments_lowercase, using original location, type {type(cmp)}")
					compartments.append(cmp)
	if not pd.isna(df.at[i,'Gene Ontology (cellular component)']):
		# process each GO term separately
		for go_term in re.sub(r'\[GO:.*?\]', '', df.at[i,'Gene Ontology (cellular component)']).strip().split('; '):
			if go_term != '':
				cmp = location_to_compartment(go_term.split('.')[0].strip())
				if isinstance(cmp, list):
					compartments.extend(cmp)
				else:
					print(f"Warning: {go_term} not found in compartments_lowercase, using original location, type {type(cmp)}")
					compartments.append(cmp)
	# if decompartmentalizing, then add any missing compartments expected by the process
	if decompartmentalize:
		sublocs_decompartmentalized = []
		for k,v in RBA_defaults.decomp_compartments.items():
			if v not in compartments and k in compartments:
				compartments.append(v)
				sublocs_decompartmentalized.append(v)
		df.at[i,'subloc_decompartmentalized'] = ','.join(sublocs_decompartmentalized)
	df.at[i,'subloc_assigned'] = ','.join(dict.fromkeys([c for c in compartments if c]))
	# ensure comments and PTM_stoich_edits columns exist
	for col in ['comments','PTM_stoich_edits']:
		if col not in df.columns:
			df[col] = ''
	# df.at[i,'subloc_assigned'] = df.at[i,'Subcellular location [CC]']
	# check for alternative products
	if not pd.isna(df.at[i,'Alternative products (isoforms)']) and df.loc[i,'Alternative products (isoforms)'] != 'ALTERNATIVE PRODUCTS:  ':
		df_debug.loc[i, ['Entry', 'Alternative products (isoforms)']] = df.loc[i, ['Entry', 'Alternative products (isoforms)']]
	ptm = str(df.loc[i,'Post-translational modification'])
	ptm_stoich = [{}]
	# Standard cleavage stoichiometries, in case explicitly cleaving the protein isn't necessary
	hamap_rule = set(re.findall(r'HAMAP-Rule:(\w+)',ptm))
	new_stoich = {}
	if hamap_rule:
		df_debug.loc[i, ['Entry', 'Post-translational modification']] = df.loc[i, ['Entry', 'Post-translational modification']]
		if hamap_rule.intersection(['MF_03209','MF_03208']): # Phosphatidylserine decarboxylase (https://doi.org/10.1016/S0005-2760(97)00101-X)
			ptm_stoich = add_metabolite_to_protein_stoich(i,ptm_stoich,coeff=1,chebi_id='28938') # NH4+
			ptm_stoich = add_metabolite_to_protein_stoich(i,ptm_stoich,coeff=-1,chebi_id='15377') # H2O
		if hamap_rule.intersection(['MF_03124']):
			ptm_stoich = add_metabolite_to_protein_stoich(i,ptm_stoich,coeff=-1,chebi_id='15377') # H2O
		if hamap_rule.intersection(['MF_03132']): # 2 serine phosphorylation instances (e.g., EC 2.7.11.1)
			new_stoich = {
				'30616':2, 		#ATP
				'456216':-2, 	#ADP
				'24636':-2, 	#H+
				}
		if hamap_rule.intersection(['MF_03048']): # thiocarboxylation (https://doi.org/10.1093/nar/gkae1111)
			new_stoich = {
				'30616':1, 		#ATP
				'456215':-1, 	#AMP
				'33019':-1, 	#PPI
				'17561':1,		#L-cys
				'16977':-1		#L-ala (dethiolated L-cys), to represent the use of cysteine desulfurase Nfs1
				}
		# if hamap_rule.intersection(['MF_03158']):

	if ptm == r"PTM: eIF-5A seems to be the only eukaryotic protein to have a hypusine residue which is a post-translational modification of a lysine by the addition of a butylamino group. {ECO:0000256|RuleBase:RU362005}.":
		# hypusine formation: https://doi.org/10.1007/s00726-021-03023-6
		new_stoich = {
			'58264':1 #4-aminobutanal, to represent the region added to Lys in DXHPSq7_m
		}
	# elif ptm == r"PTM: Binds 1 heme group per subunit. {ECO:0000256|RuleBase:RU004427}.":
	# 	new_stoich = {
	# 		'60562':1 #ferroheme c; could also represent via ferrocytochrome c metabolite, but it's often only oxidized or reduced in GSM models, rarely formed in its entirety. Including a sink or source thus risks upsetting the model's redox balance; be cautious when including.
	# 	}
	elif ptm == r"PTM: Topaquinone (TPQ) is generated by copper-dependent autoxidation of a specific tyrosyl residue. {ECO:0000256|PIRSR:PIRSR600269-51, ECO:0000256|RuleBase:RU000672}.":
		# topaquinone mechanism from https://doi.org/10.1007/s007750050283 and https://doi.org/10.1021/bi0504759
		new_stoich = {
			'16240':-1, #H2O2
			'15379':1 #O2
		}
	# if ptm == r"PTM: Transiently phosphorylated on a His residue during the reaction cycle. Phosphorylation strongly increases the affinity for substrates and increases the rate of nicotinate D-ribonucleotide production. Dephosphorylation regenerates the low-affinity form of the enzyme, leading to product release. {ECO:0000256|RuleBase:RU003838}.":
		# ignore unless NAMNPP_c or other relevant rxns don't implicitly account for ATP hydrolysis (e.g., via including H+ as reactant), since the phosphate released via hydrolysis is from the enzyme
	# if ptm == r"PTM: During the catalytic reaction, a sulfide is transferred from Cys-231 to a reaction intermediate, generating a dehydroalanine residue. {ECO:0000256|HAMAP-Rule:MF_03158}.":
		# ignore since it's consumed in the rxn itself, so a GSM model including it should already have L-cys consumed when using it
	# update PTM stoich for more complex entries; only include the whole stoich if it's all present in the model, to avoid mass balance issues
	if all(chebi_in_model(id) for id in new_stoich.keys()):
		for id,c in new_stoich.items():
			ptm_stoich = add_metabolite_to_protein_stoich(i,ptm_stoich,coeff=c,chebi_id=id)

	# print(re.findall(r'CHEBI:(\d+)',str(df.loc[i,'Cofactor'])))
	# find all "COFACTOR: " statements (case-sensitive) and extract CHEBI IDs. Each statement may have multiple CHEBI IDs separated by commas, indicating they are alternatives.
	# process each "COFACTOR: " statement separately
	for cofactor_statement in re.findall(r'\bCOFACTOR:(.*?)(?=\bCOFACTOR:|$)', str(df.loc[i,'Cofactor'] if pd.notna(df.loc[i,'Cofactor']) else '')):
		cofactor_options = cofactor_statement.split(' Name=')[1:] # separate each cofactor, removing the first 'COFACTOR: ' part
		cof_dict = {}

		cofactor_subunit_multiplier = 1
		for cof in cofactor_options:
			name = cof.split(';')[0]
			cof_dict[name] = {'chebi':cof.split('Xref=ChEBI:CHEBI:')[1].split(';')[0],'evidence':cof.split('Evidence=')[1].split(';')[0],'n':1}
			if 'Note=' in cof:
				cof_dict[name]['note'] = cof.split('Note=')[1].split(';')[0]
				# check if 'Binds ' is in the note
				binding_regex = re.search(r'(Binds|Expected to bind|Probably binds)[^\d]*\s(\d+|one|two|three|four|five|six|seven|eight|nine)', cof_dict[name]['note'])
				# n = cofactors/unit (where unit is e.g., subunit, dimer, etc.)
				# if binding_regex returns a number word, convert it to an integer
				cof_dict[name]['n'] = 1
				if binding_regex:
					number_word_map = {
						'one': 1,
						'two': 2,
						'three': 3,
						'four': 4,
						'five': 5,
						'six': 6,
						'seven': 7,
						'eight': 8,
						'nine': 9
					}
					possible_number = binding_regex.group(2)
					if possible_number.isdigit():
						cof_dict[name]['n'] = int(possible_number)
					elif possible_number.lower() in number_word_map:
						cof_dict[name]['n'] = number_word_map[possible_number.lower()]
					else:
						cof_dict[name]['n'] = 1
				# check if there's a ' per ' after the number (e.g., "Binds 2 per subunit")
				# per_unit_regex = re.search(r'(?<= per ).*?(?= per|$)', cof_dict[name]['note'])
				# if binding_regex and per_unit_regex:
				# 	# per_unit = cofactor unit. 
				# 	per_unit = get_mer(df.loc[i],str_to_check=per_unit_regex.group(0),column_to_check=None)
				# 	# for adjusting n to cofactors/copy, need to multiply by units/copy. (e.g., if protein is cleaved into 2 subunits, and 3 cofactors bind per subunit, then cofactors/copy = 3 * (2 subunits/copy) = 6 cofactors/copy)
				# 	cofactor_subunit_multiplier = df.loc[i,'subunits/copy'] / per_unit
				# 	if cofactor_subunit_multiplier != 1:
				# 		print(f"Adjusting cofactor {name} binding for protein {df.loc[i,'Entry']}: {cof_dict[name]['n']} cofactors per {per_unit} units; subunits/copy = {df.loc[i,'subunits/copy']}, so cofactor_subunit_multiplier = {cofactor_subunit_multiplier}")
				# 	# adjust n to cofactors/copy: cofactors/copy = (cofactors/unit * unit/copy)
				# 	# cof_dict[name]['n'] = df.loc[i,'subunits/copy'] * cof_dict[name]['n'] / per_unit
					
		cof_count_max = max(cof_dict[cof]['n'] for cof in cof_dict)
		usable_cofactor_found = False
		for cof in cof_dict.copy():
			cof_dict[cof]['n'] = cof_count_max
			id = cof_dict[cof]['chebi']
			# remove unknown cofactors or replace them if matches are found
			if not chebi_in_model(id):
				# search ChEBI
				match_found = False
				chebi_children = set()
				# search ChEBI and handle network/JSON errors gracefully
				try:
					chebi_data = search_chebi(id, False)
				except requests.exceptions.RequestException as e:
					warn(f"Error querying ChEBI for ID {id}: {e}")
					chebi_data = {}
				except Exception as e:
					# catch JSONDecodeError and any other unexpected issues
					warn(f"Failed to parse ChEBI response for ID {id}: {e}")
					chebi_data = {}
				
				# search parent IDs to see if they're present in the model
				chebi_ontology = chebi_data.get('ontology_relations',{})
				for parent in chebi_ontology.get('outgoing_relations',[]):
					# repeat search_chebi but w/o searching children (since parent terms can be very broad)
					_ = search_chebi(parent.get('final_id',''),False,False,False)
					if chebi_in_model(parent.get('final_id','')):
						warn(f"Cofactor with ChEBI ID {id} has parent ID {parent.get('final_id','')} in model.")
				
				if not match_found:
					cof_dict.pop(cof)
		# check if any cofactors can be used
		for cof in cof_dict.copy():
			id = cof_dict[cof]['chebi']
			if not model_reduced:
				usable_cofactor_found = True
				continue
			# if cofactor in model_reduced, set usable_cofactor_found to True to indicate that this cofactor can be used in the PTM stoichiometry
			if id in model_reduced_chebi_ids:
				usable_cofactor_found = True
		if len(cof_dict.keys()) == 0:
			# these may be special cases where cofactors are mentioned without CHEBI IDs, requiring manual curation (e.g., A9CEQ7)
			df.loc[i,'comments'] = df.loc[i,'Cofactor']
			continue
		elif len(cof_dict.keys()) > 1:
			# if multiple cofactors are listed, they are alternatives; create a new stoichiometry dictionary in ptm_stoich for each combination (e.g., if 2 cofactors (A and B) are listed, and n=2 for 1 of them, make versions w/ 2A and 2B, and 1A+1B)
			if any(cof_dict[cof]['n'] > 1 for cof in cof_dict):
				# if any cofactor has n>1, create combinations accordingly
				new_ptm_stoich = []
				for stoich in ptm_stoich:
					# create combinations of cofactors based on their n values
					cof_names = list(cof_dict.keys())
					n_values = [cof_dict[cof]['n'] for cof in cof_names]
					# generate all combinations of cofactors based on their n values
					ranges = [range(n+1) for n in n_values]
					for counts in product(*ranges):
						if sum(counts) != cof_count_max:
							continue # skip the case where no cofactors are added
						new_stoich = stoich.copy()
						for cof_name, count in zip(cof_names, counts):
							if count > 0:
								chebi_id = cof_dict[cof_name]['chebi']
								new_stoich = add_metabolite_to_protein_stoich(i,new_stoich,coeff=count,chebi_id=chebi_id)
						if type(new_stoich) is list:
							new_ptm_stoich.extend(new_stoich)
						else:						
							new_ptm_stoich.append(new_stoich)
			else:
				# if all n=1, create a new stoichiometry dictionary for each cofactor
				new_ptm_stoich = []
				for stoich in ptm_stoich:
					for cof_name in cof_dict:
						chebi_id = cof_dict[cof_name]['chebi']
						new_stoich = add_metabolite_to_protein_stoich(i,stoich.copy(),coeff=1,chebi_id=chebi_id)
						if type(new_stoich) is list:
							new_ptm_stoich.extend(new_stoich)
						else:
							new_ptm_stoich.append(new_stoich)
			for new_stoich_item in new_ptm_stoich:
				if new_stoich_item == dict():
					new_ptm_stoich.remove(new_stoich_item)
			ptm_stoich = new_ptm_stoich if new_ptm_stoich != [] else ptm_stoich
		else:
			# if only 1 cofactor is listed, add it to all existing stoichiometry dictionaries in ptm_stoich
			cof_name = list(cof_dict.keys())[0]
			chebi_id = cof_dict[cof_name]['chebi']
			ptm_stoich = add_metabolite_to_protein_stoich(i,ptm_stoich,coeff=cof_dict[cof_name]['n'],chebi_id=chebi_id)
		if not usable_cofactor_found:
			warn(f"No usable cofactors found in model for protein {df.loc[i,'Entry']} at binding site with this cofactor statement: {cofactor_statement}",ignore_duplicate_warnings=True)
		# update ptm_stoich to account for cofactor_subunit_multiplier; do this after all cofactors are added to avoid complications with alternative combinations (e.g., 1 cofactor/dimer).
		if cofactor_subunit_multiplier != 1:
			for stoich in ptm_stoich:
				for met in stoich.keys():
					stoich[met] = int(stoich[met] * cofactor_subunit_multiplier)
	# If 'EC number' column is 1.5.5.2, and FAD (CHEBI:57692) is already a cofactor, remove it to avoid redundancy
	if pd.notna(df.loc[i,'EC number']) and df.loc[i,'EC number'] == '1.5.5.2':
		cofactor_ids_to_check = met_chebi_map.get('57692',{}).values()
		# print(f"Checking for FAD in reactions for protein {df.loc[i,'Entry']} with EC number")
		# if FAD is present in all rxns for this enzyme, remove it from the PTM_stoich
		prot_rxns = model.genes.get_by_id(df.loc[i,'gene_src']).reactions if pd.notna(df.loc[i,'gene_src']) else []
		# print(f"Found {len(prot_rxns)} reactions for protein {df.loc[i,'Entry']} with gene source {df.loc[i,'gene_src']}")
		if all([any(cofactor in [m.id for m in r.metabolites.keys()] for cofactor in cofactor_ids_to_check) for r in prot_rxns]):
			for stoich in ptm_stoich:
				for met in cofactor_ids_to_check:
					stoich.pop(met, None)
	# convert the stoichiometries to a string, joining the keys and values with colons (and removing unnecessary spaces)
	if ptm_stoich != [{}]:
		# remove duplicates in PTM_stoich, as well as cofactor combinations that are supersets of others
		ptm_stoich = list(dict.fromkeys(tuple(tuple(sorted(i.items())) for i in ptm_stoich)))
		ptm_stoich = [dict(stoich) for stoich in ptm_stoich]
		
		for stoich in ptm_stoich.copy():
			# remove any entries with a coefficient of 0
			for met in list(stoich.keys()):
				if stoich[met] == 0:
					del stoich[met]
		for stoich in ptm_stoich.copy():
			# remove it if it's a superset of other cofactor combos, since the model favors using fewer cofactors
			ptm_stoich_copy = ptm_stoich.copy()
			ptm_stoich_copy.remove(stoich)
			for other_stoich in ptm_stoich_copy:
				is_superset = True
				for met in other_stoich.keys():
					if met in stoich.keys():
						if int(stoich[met]) < int(other_stoich[met]):
							is_superset = False
					else:
						is_superset = False
				if is_superset:
					ptm_stoich.remove(stoich)
					break
			else:
				continue
				
		df.loc[i,'PTM_stoich_edits'] = '|'.join([
			','.join([f'{k}:{v}' for k,v in stoich.items()]) for stoich in ptm_stoich if stoich != {}
		])
	# if ptm_stoich:
	# 	df_debug.loc[i, ['Entry', 'PTM_stoich_edits']] = df.loc[i, ['Entry', 'PTM_stoich_edits']]
	df.loc[i,'manually reviewed'] = ''

df['MW (g/mmol)'] = protein_mw_from_sequence(df['sequence'].fillna('').tolist(), path='input/PROTEIN_amino_acid_map.txt')
# remove subunits/copy column
df = df.drop(columns=['subunits/copy'])

CHEBI ID 49883 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 26930 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33725 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33723 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33722 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33724 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 24875 not found in met_chebi_map when updating protein stoichiometry


/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_4547/2387424365.py:355: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'A0A099NUE6' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_debug.loc[i, ['Entry', 'Post-translational modification']] = df.loc[i, ['Entry', 'Post-translational modification']]
/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_4547/2387424365.py:355: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'PTM: C-terminal thiocarboxylation occurs in 2 steps, it is first acyl-adenylated (-COAMP) via the hesA/moeB/thiF part of UBA4, then thiocarboxylated (-COSH) via the rhodanese domain of UBA4. {ECO:0000256|HAMAP-Rule:MF_03048}.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_debug.loc[i, ['Entry', 'Post-translational modificatio

CHEBI ID 190135 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33739 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33738 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33737 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 60240 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 33006 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 49786 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 37317 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 37286 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 30502 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 37136 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 35104 not found in met_chebi_map when updating protein stoichiometry
CHEBI ID 49978 not found in met_chebi_map when updating protein

/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_4547/2387424365.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'COFACTOR: Name=heme b; Xref=ChEBI:CHEBI:60344; Evidence={ECO:0000256|RuleBase:RU362117}; Note=Binds 2 heme groups non-covalently. {ECO:0000256|RuleBase:RU362117}; COFACTOR: Name=heme; Xref=ChEBI:CHEBI:30413; Evidence={ECO:0000256|PIRSR:PIRSR038885-2}; Note=Binds 2 heme groups non-covalently. {ECO:0000256|PIRSR:PIRSR038885-2};' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_debug.loc[pro, ['Entry', 'Cofactor']] = df.loc[pro, ['Entry', 'Cofactor']]


In [20]:
# print the entries, "cofactor" statements, and PTM_stoich_edits columns for the proteins where PTM_stoich_edits is not empty
df_ptms = df[df['PTM_stoich_edits'].notna() & (df['PTM_stoich_edits'] != '')][['Entry','Cofactor','PTM_stoich_edits']]
if not df_ptms.empty:
	display(df_ptms)

,Entry,Cofactor,PTM_stoich_edits
468,A0A099NP52,COFACTOR: Name=FAD; Xref=ChEBI:CHEBI:57692; Ev...,fad_c:1
621,A0A099NQC0,COFACTOR: Name=Mg(2+); Xref=ChEBI:CHEBI:18420;...,mg2_c:1
694,A0A099NQP4,COFACTOR: Name=Fe(2+); Xref=ChEBI:CHEBI:29033;...,fe2_c:1
698,A0A099NQP8,COFACTOR: Name=FAD; Xref=ChEBI:CHEBI:57692; Ev...,fad_c:1
728,A0A099NQT4,COFACTOR: Name=Fe(2+); Xref=ChEBI:CHEBI:29033;...,fe2_c:1
...,...,...,...
6866,A0A099PA07,COFACTOR: Name=pyridoxal 5'-phosphate; Xref=Ch...,pydx5p_c:1
6868,A9YUC7,COFACTOR: Name=heme; Xref=ChEBI:CHEBI:30413; E...,hemeA_c:1|hemeO_c:1|sheme_c:1
6873,P40011,COFACTOR: Name=Ni(2+); Xref=ChEBI:CHEBI:49786;...,cobalt2_c:1|zn2_c:1
6878,A0A2U9RAR9,COFACTOR: Name=Cu cation; Xref=ChEBI:CHEBI:233...,cu2_m:1|cu2_mm:1


In [21]:
# for all proteins in cofactor_modifications, implement the specified modifications to the cofactor assignments in the DataFrame, and add comments to the 'comments' column to indicate the changes made
df_temp = deepcopy(df)
# group all proteins by 'pro' in cofactor_modifications
proteins_to_modify = set(mod['pro'] for mod in cofactor_modifications)
for pro in proteins_to_modify:
	pro_mods = [mod for mod in cofactor_modifications if mod['pro'] == pro]
	comments = []
	idx = df.index[df['Entry'] == pro][0]
	ptm_stoich_dict = {}
	# Make 'default' key if not present
	if type(df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0]) == str and df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0] != '':
		ptm_stoich_dict['default'] = df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0]
	else:
		ptm_stoich_dict = df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0] if isinstance(df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0], dict) else {}
	for mod in pro_mods:
		met = mod['met']
		comment = mod['mod']
		# check if model.metabolites.get_by_id(met).compartment is in PTM_stoich_edits already
		# if type(df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0]) != dict or model.metabolites.get_by_id(met).compartment not in df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0].keys():
		if model.metabolites.get_by_id(met).compartment not in ptm_stoich_dict.keys():
			ptm_stoich_dict[model.metabolites.get_by_id(met).compartment] = ptm_stoich_dict['default'] if 'default' in ptm_stoich_dict else df.loc[df['Entry'] == pro, 'PTM_stoich_edits'].values[0]
		if comment == 'remove from other compartments':
			# remove the specified metabolite from all other compartments
			for key in sorted(ptm_stoich_dict.keys()):
				if key == model.metabolites.get_by_id(met).compartment:
					continue
				# convert the 'default' key's value from a string to a dict with metabolite IDs as keys and coefficients as values
				ptm_stoich_default = []
				for item in sorted(ptm_stoich_dict[key].split('|')):
					stoich_dict = {}
					for met_coeff in item.split(','):
						if met_coeff != '':
							met_id, coeff = met_coeff.split(':')
							stoich_dict[met_id] = int(coeff)
					if stoich_dict != {} and met not in stoich_dict:
						# print(f"Processing item '{item}' for key '{key}' in PTM stoichiometry edits for protein {pro} to remove metabolite {met} from other compartments. Stoichiometry dict before removal: {stoich_dict}")
						ptm_stoich_default.append(stoich_dict)
				# convert back to string
				ptm_stoich_dict[key] = '|'.join([','.join([f'{k}:{v}' for k,v in stoich.items()]) for stoich in ptm_stoich_default])
	# find df entry where entry matches pro, and update PTM_stoich_edits column with the new ptm_stoich_dict (converted to string)
	print(f"Updating PTM stoichiometry edits for protein {pro} to remove metabolite {met} from other compartments. New PTM stoichiometry dict: {ptm_stoich_dict}")
	# if there are no non-blank values in ptm_stoich_dict, make the value an empty string instead of a dict with empty values
	if all(value == '' for value in ptm_stoich_dict.values()):
		ptm_stoich_dict = ''
	df_temp.at[idx, 'PTM_stoich_edits'] = ptm_stoich_dict
		
if locations_unmapped:
	print(f"Warning: the following locations were not mapped to compartments: {locations_unmapped}")
# make all PTM_stoich_edits entries that are NaN into empty strings
df_temp['PTM_stoich_edits'] = df_temp['PTM_stoich_edits'].fillna('')
# output the draft
protein_stoich_path = './input/PROTEIN_stoich_curation.tsv'
df_temp.to_csv(protein_stoich_path, index=False, sep='\t')

# save the dictionary to a file
with open('./input/met_chebi_map.json','w') as f:
	json.dump(met_chebi_map,f,indent=4)

Updating PTM stoichiometry edits for protein A0A099P3E6 to remove metabolite fad_m from other compartments. New PTM stoichiometry dict: {'m': ''}
Updating PTM stoichiometry edits for protein A0A099P470 to remove metabolite fad_m from other compartments. New PTM stoichiometry dict: {'default': 'fad_c:1', 'm': 'fad_m:1|fad_c:1'}
Updating PTM stoichiometry edits for protein A0A099NYV0 to remove metabolite mn2_rm from other compartments. New PTM stoichiometry dict: {'default': 'ca2_c:1,mg2_c:1,mn2_c:1', 'rm': 'ca2_rm:1,mg2_c:1,mn2_c:1|ca2_c:1,mg2_c:1,mn2_c:1|ca2_rm:1,mg2_c:1,mn2_rm:1|ca2_c:1,mg2_c:1,mn2_rm:1'}
Updating PTM stoichiometry edits for protein A0A099NUM6 to remove metabolite fad_m from other compartments. New PTM stoichiometry dict: {'default': 'fad_c:1', 'm': 'fad_m:1|fad_c:1'}
Updating PTM stoichiometry edits for protein A0A099NZ20 to remove metabolite fad_m from other compartments. New PTM stoichiometry dict: {'default': 'fad_c:1', 'm': 'fad_m:1|fad_c:1'}
Updating PTM stoichi

In [22]:
df = deepcopy(df_temp)

In [23]:
cofactor_modifications

[{'pro': 'A0A099NUM6',
  'met': 'fad_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NUM7',
  'met': 'cu2_mm',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NUP4',
  'met': 'thmpp_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NV29',
  'met': 'fe2_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NV29',
  'met': 'cu2_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NWG1',
  'met': 'fad_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NWV8',
  'met': 'fad_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NXE1',
  'met': 'h2o_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NXK9',
  'met': 'cu2_mm',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NY51',
  'met': 'zn2_e',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NYS9',
  'met': 'fmn_m',
  'mod': 'remove from other compartments'},
 {'pro': 'A0A099NYV0',
  'met': 'ca2_rm',
  'mod': 'remove fr

In [24]:
chebi_data_cache

{'49883': {'id': 49883,
  'chebi_accession': 'CHEBI:49883',
  'name': 'tetra-μ<small><sub>3</sub></small>-sulfido-tetrairon',
  'stars': 3,
  'definition': 'An Fe<small><sub>4</sub></small>S<small><sub>4</sub></small> iron-sulfur cluster in which each sulfur is attached to three of the iron atoms.',
  'ascii_name': 'tetra-mu3-sulfido-tetrairon',
  'names': {'UNIPROT NAME': [{'name': '[4Fe-4S] cluster',
     'status': 'C',
     'type': 'UNIPROT NAME',
     'source': 'UniProt',
     'ascii_name': '[4Fe-4S] cluster',
     'adapted': False,
     'language_code': 'en'}],
   'SYNONYM': [{'name': '[4Fe-4S] cluster',
     'status': 'C',
     'type': 'SYNONYM',
     'source': 'IUBMB',
     'ascii_name': '[4Fe-4S] cluster',
     'adapted': False,
     'language_code': 'en'},
    {'name': '[Fe<small><sub>4</sub></small>S<small><sub>4</sub></small>]',
     'status': 'C',
     'type': 'SYNONYM',
     'source': 'ChEBI',
     'ascii_name': '[Fe4S4]',
     'adapted': False,
     'language_code': 'en'}

In [25]:
# get CHEBI IDs from BiGG (if available)
chebi_ontology_dict = {}
chebi_parents = set()
chebi_children = set()
chebi_children_of_concern = set(met_chebi_map.keys()).update(chebi_ids_ambiguous)
for id in chebi_ids_unknown:
	if id in chebi_ids_ambiguous:
		warn(f'CHEBI ID {id} is potential match for model metabolite(s) and cofactor(s); consider adding it to the GSM model and rerunning this script.')
		continue

In [26]:
#http://bigg.ucsd.edu/api/v2/universal/metabolites/
chebi_ids_unknown

{'10113',
 '10744',
 '13082',
 132253,
 '13319',
 '13320',
 '13321',
 '13357',
 '13370',
 '13379',
 '13383',
 '134067',
 '13861',
 '14386',
 '14519',
 '14957',
 16793,
 '17208',
 17627,
 176783,
 '190135',
 '20882',
 '20883',
 '21129',
 '21130',
 '21134',
 '21137',
 '21139',
 '21240',
 '21241',
 '21435',
 '21436',
 22465,
 '22534',
 22651,
 22896,
 '22988',
 231854,
 '23236',
 23336,
 '23337',
 '23338',
 23357,
 23378,
 '23378',
 '23379',
 '23380',
 '23381',
 '23856',
 24008,
 24481,
 24482,
 24483,
 24484,
 24485,
 24486,
 24487,
 '24488',
 '24491',
 '24493',
 24636,
 '24636',
 '24875',
 24875,
 '24876',
 '24877',
 '24878',
 '25056',
 '25112',
 25155,
 '25156',
 '25157',
 25158,
 '25199',
 '25200',
 25212,
 '25213',
 25213,
 25216,
 25516,
 '25517',
 '25518',
 '26219',
 26348,
 26355,
 '26619',
 '26690',
 '26717',
 26930,
 27141,
 27142,
 27143,
 27147,
 27148,
 '27368',
 28599,
 '2868',
 29037,
 29041,
 29073,
 29120,
 29233,
 29234,
 '29787',
 '30051',
 '30143',
 '30179',
 30313,
 3

In [27]:
df_debug

,Entry,Post-translational modification,Cofactor
1934,A0A099NUE6,PTM: C-terminal thiocarboxylation occurs in 2 ...,NaN
2952,A0A099NXE1,PTM: The alpha and beta chains are autoproteol...,NaN
3396,A0A099NYN7,"PTM: During the catalytic reaction, a sulfide ...",NaN
4111,A0A099P0Q7,PTM: Phosphorylation at Ser-174 and Ser-175 pr...,NaN
5911,A0A099P5W8,PTM: Is synthesized initially as an inactive p...,NaN
6883,A0A2U9RBK5,NaN,COFACTOR: Name=heme b; Xref=ChEBI:CHEBI:60344;...


In [28]:
df = pd.read_csv(protein_stoich_path, sep='\t')

# check "Gene Names" column (space-separated) for genes in model
for i in df.index:
	if pd.isna(df.loc[i, 'Gene Names']) or not df.loc[i, 'Gene Names']:
		df.loc[i, 'gene_src'] = df.loc[i, 'Gene Names'].apply(
			lambda x: next((gene for gene in x.split() if gene in [g.id for g in model.genes]), None)
		)

# automatically determine gene_src by comparing uniprot IDs to PROTEIN_stoich_curation.tsv, if available
# if os.path.exists('input/PROTEIN_stoich_curation.tsv'):
#     df_cur = read_spreadsheet('input/PROTEIN_stoich_curation.tsv')
#     # Drop duplicate uniprot entries to ensure unique index for mapping
#     df_cur_unique = df_cur.drop_duplicates(subset='uniprot', keep='first')
#     df['gene_src'] = df['uniprot'].map(df_cur_unique.set_index('uniprot')['id'])
# set gene_src to the index of df
df.index = df['gene_src']
df
# save the updated DataFrame back to the file
df.to_csv(protein_stoich_path, index=False, sep='\t')

In [29]:
# Enzyme stoich drafting
df_pro = df.copy()

whole_biomass_rxns = list(dict.fromkeys(['BIOSYN-' + i for i in ['BIODILAERO', 'BIODILAERO-NOGAM','BIOXYL-PINHEIRO','BIOXYL-PINHEIRO-NOGAM']] + ['BIOMASS_xyl_hybrid',RBA_defaults.RBA_biomass_id,RBA_defaults.SM_biomass_id]))

# checks which rxns to ignore when seeing if all necessary enzymes are modeled. Ignore blocked rxns (to avoid unnecessary work from adding them)
SM_biomass_rxns = [rxn.id for rxn in find_biomass_reactions(model)] + ['BIOMASS_MFA','BIOMASS_MFA_NO_GAM','BIOMASS_RBA']
rxns_not_needing_enzymes = list(dict.fromkeys([r.id for r in model.reactions if r.bounds == (0,0)] + SM_biomass_rxns))
# Gene-Protein Reaction associations (GPRs) representing spontaneous reactions (i.e., no enzyme needed in model)
spont_GPRs = RBA_defaults.spont_GPRs
# take IDs of all rxns
print('Reactions to ignore when checking if enzymes must be added:')
print(rxns_not_needing_enzymes)
# Expanding df_pro to include sublocations as separate proteins
# if the 'gene_src' column is empty, fill it with the 'name' column
if 'name' in df_pro.columns:
	df_pro['gene_src'] = df_pro['gene_src'].fillna(df_pro['name'])
## for each protein, if there are commas in subloc_assigned, split them into separate entries with 'id' modified to reflect sublocation
df_pro['subloc_assigned'] = df_pro['subloc_assigned'].apply(lambda x: x.split(',') if type(x) == str and ',' in x else x)
# add the 'id' column back in, assigning it to "gene_src" plus "subloc_assigned" if the latter is not 'unknown'
# modify IDs to reflect sublocation
# add location only if there are multiple proteins with the same gene source
df_pro = df_pro.explode('subloc_assigned').reset_index(drop=True)
if 'id' not in df_pro.columns:
	has_entry = 'Entry' in df_pro.columns
	has_protein_col = 'protein' in df_pro.columns
	df_pro['id'] = df_pro.apply(
		lambda row: (
			str(row['gene_src']) if pd.notna(row['gene_src'])
			else str(row['Entry']) if has_entry and pd.notna(row['Entry'])
			else str(row['protein']) if has_protein_col and pd.notna(row['protein'])
			else str(row.name)
		),
		axis=1
	)
	# add sublocation only if multiple proteins have the same ID (for continuity w/ past RBA versions; unclear if needed, though having consistent IDs would help futureproof other scripts)
	df_pro['id'] = df_pro.apply(
		lambda row: (
			str(row['id'])
		)
		+ (
			'_' + row['subloc_assigned']
			if pd.notna(row['subloc_assigned'])
			and row['subloc_assigned'] != 'unknown'
			and df_pro[df_pro['id'] == row['id']].shape[0] > 1
			else ''
		),
		axis=1
	)
df_pro.index = df_pro.id.to_list()
# calculate MW of enzymes = sum of MW of all proteins in protein_stoich (MW from df_pro) * number of protein_stoich

# list all genes, to map them to translation (PROSYN) rxns and other rxns making their products where needed
# combine all genes from df_pro and model
# combine all genes from df_pro and model, ensuring all are strings and not NaN
genes_model = [str(gene.id) for gene in model.genes if pd.notna(gene.id)]
genes_pro = [str(g) for g in df_pro.gene_src.to_list() if pd.notna(g)]
genes = sorted(list(dict.fromkeys(genes_model + genes_pro)))
gene_expression_dict = {}
#### Assemble reactions
# Automatically create placeholders for enzymes based on GPRs in model.reactions
rxns_to_skip = set(rxns_not_needing_enzymes)
enz_rxn_dict = {}
spont_rxn_dict = {}
rxn_updates = {rxn.id: {} for rxn in model.reactions}
GPR_enz_dict = {}
mw = 'MW (g/mmol)'
df_enz_rxns = pd.DataFrame(columns=['enz','rxns','protein_stoich','compartment-protein_maps',mw])
# add dict of protein_stoich to enz_rxn_dict, with each protein as a key and its stoichiometry as the value

# find all unique GPRs in model.reactions, and their associated reactions
proteins_with_location_conflicts = {}
for rxn in model.reactions:
	# print(rxn.id, rxn.gene_reaction_rule)
	if rxn.id in rxns_to_skip:
		continue
	if rxn.gene_reaction_rule not in spont_GPRs + ['']:
		if rxn.gene_reaction_rule in GPR_enz_dict.keys():
			# add rxn to corresponding enz in enz_rxn_dict
			enz_ids = GPR_enz_dict[rxn.gene_reaction_rule]
			for enz_id in enz_ids:
				enz_rxn_dict[enz_id]['rxns'].append(rxn.id)
				rxn_updates[rxn.id].setdefault('enz',enz_id)
		else:
			GPR_enz_dict[rxn.gene_reaction_rule] = set()
			enzs = sorted(find_true_combinations(rxn.gene_reaction_rule, use_knockout=True, variables=[g.id for g in rxn.genes]), key=lambda x: sorted(x.keys()))
			for enz in enzs:
				# print('\t',enz)
				enz_found = None
				# default: 1 copy of each protein is used
				# prot_stoich = ','.join([gene+':1' for gene in sorted(enz.keys()) if enz[gene]])
				# assemble prot_stoich as a dict with proteins as keys and their stoichiometry as values ('stoich' column of df)
				prot_stoich = {}
				for gene in sorted(enz.keys()):
					if enz[gene] and gene not in spont_GPRs:
						pro_matches = None
						# if gene is in df_pro, use its stoichiometry, otherwise use 1
						if gene in df_pro.index:
							pro_matches = df_pro.loc[df_pro.index == gene]
							prot_stoich_val = pro_matches['stoich'].iloc[0]
							# Remove unnecessary decimals: if value is integer-like, cast to int
							if isinstance(prot_stoich_val, float) and prot_stoich_val.is_integer():
								prot_stoich_val = int(prot_stoich_val)
							prot_stoich[gene] = prot_stoich_val
						elif gene in df_pro.gene_src.to_list():
							# get the first matching protein's stoichiometry
							pro_matches = df_pro.loc[df_pro['gene_src'] == gene]
							if not pro_matches.empty:
								prot_stoich_val = pro_matches['stoich'].iloc[0]
								# Remove unnecessary decimals: if value is integer-like, cast to int
								if isinstance(prot_stoich_val, float) and prot_stoich_val.is_integer():
									prot_stoich_val = int(prot_stoich_val)
								prot_stoich[gene] = prot_stoich_val
							else:
								# if no matches found, use 1 as default stoichiometry
								prot_stoich[gene] = 1
						else:
							# if gene is not in df_pro, use 1 as default stoichiometry
							prot_stoich[gene] = 1
						# check if the locations in pro_matches include the compartment of the reaction
						if pro_matches is not None:
							compartments_prot = set()
							for loc in pro_matches['subloc_assigned']:
								if pd.notna(loc):
									compartments_prot.update(loc.split(','))
							if rxn.compartments not in (None, '', set()) and not compartments_prot.intersection(set(rxn.compartments)):
								# if no evidence from database suggesting specific compartments, include all GSM rxn compartments to aim for more optimistic modeling
								if all(pd.isna(pro_matches[col]).all() for col in ['Subcellular location [CC]','Gene Ontology (cellular component)']):
									for prot in pro_matches.index:
										df.at[prot,'subloc_assigned'] = ','.join(dict.fromkeys(list(compartments_prot) + list(rxn.compartments)))
								else:
									# print(f'Warning: protein {gene} with locations {compartments_prot} may not be in the same compartment as reaction {rxn.id} ({rxn.compartments})')
									proteins_with_location_conflicts.setdefault(gene, set()).update(rxn.compartments)

				# check if any enzymes with the same stoichiometry are already in enz_rxn_dict
				enz_id = ''.join(sorted(prot_stoich.keys()))

				for old_enz_id in enz_rxn_dict:
					# print(enz_id,enz_rxn_dict[enz_id])
					# automatically assumes enzymes are the same if they have the same proteins, regardless of order/count
					if sorted(enz_rxn_dict[old_enz_id]['protein_stoich'].keys()) == sorted(prot_stoich.keys()):
						enz_rxn_dict[old_enz_id]['rxns'].append(rxn.id)
						enz_found = old_enz_id
						break
				if not enz_found:
					# enz_rxn_dict[rxn.id] = enz
					enz_rxn_dict[enz_id] = {'rxns':[rxn.id],
											'protein_stoich':prot_stoich}
					rxn_updates[rxn.id].setdefault('enz',enz_id)
					GPR_enz_dict[rxn.gene_reaction_rule].add(enz_id)
				else:
					GPR_enz_dict[rxn.gene_reaction_rule].add(enz_found)
	else:
		# add to spont_rxn_dict under its GPR as a list of rxn IDs, if it is not already there
		spont_rxn_dict[rxn.gene_reaction_rule] = spont_rxn_dict.get(rxn.gene_reaction_rule, {'rxns':[],'protein_stoich':{'zeroCost':1}})
		spont_rxn_dict[rxn.gene_reaction_rule]['rxns'].append(rxn.id)
		rxn_updates[rxn.id].setdefault('gene_reaction_rule',rxn.gene_reaction_rule)
		# spont_rxn_dict[rxn.gene_reaction_rule] = spont_rxn_dict.get(rxn.gene_reaction_rule, [rxn.id])
		# spont_rxn_dict[rxn.gene_reaction_rule] = rxn.id

# reduce enzyme names to fit within GAMS limits while keeping them unique and informative.
max_rxn_id_length = max_id_length - 13 # GAMS has a limit of 63 characters for variable names, but reserving some characters for prefixes/suffixes (8 for "ENZLOAD-", 5 for "_FWD-" or "_REV-")
enz_rxn_dict_updated = deepcopy(enz_rxn_dict)

def find_shortest_unique_substring(names):
	substring_dict = {}
	for name in names:
		for i in range(len(name)):
			for j in range(i+1, len(name)+1):
				substring = name[i:j]
				if substring in substring_dict:
					substring_dict[substring].add(name)
				else:
					substring_dict[substring] = {name}
	# find the shortest unique substring for each name
	unique_substring_dict = {}
	for name in names:
		unique_substring = None
		for i in range(len(name)):
			for j in range(i+1, len(name)+1):
				substring = name[i:j]
				if len(substring_dict[substring]) == 1:
					if unique_substring is None or len(substring) < len(unique_substring):
						unique_substring = substring
		unique_substring_dict[name] = unique_substring if unique_substring is not None else name
	return unique_substring_dict

def update_GPR_enz_dict(old_enz_id, new_enz_id):
	for gpr, enzs in GPR_enz_dict.items():
		if old_enz_id in enzs:
			enzs.remove(old_enz_id)
			enzs.add(new_enz_id)

# find shortest rxn name associated with each enzyme in enz_rxn_dict, and use it to create a shortened enzyme name that fits within GAMS limits
for enz in enz_rxn_dict:
	if len(enz) < max_rxn_id_length: # only update names that are too long, to preserve readability as much as possible
		continue
	rxns = enz_rxn_dict[enz]['rxns']
	# find all rxns for which this enzyme is the only associated enzyme, and base the new enzyme ID on the shortest name among them
	rxns_with_only_this_enz = [rxn for rxn in rxns if len([e for e in enz_rxn_dict if rxn in enz_rxn_dict[e]['rxns']]) == 1]
	best_rxn_id = min(rxns_with_only_this_enz, key=len) if rxns_with_only_this_enz else min(rxns, key=len)
	longest_rxn_id = max(rxns, key=len)
	if len(longest_rxn_id + best_rxn_id) > max_rxn_id_length:
		print(f'Warning: longest rxn ID {longest_rxn_id} combined with shortest unique rxn ID {best_rxn_id} is still too long for GAMS when combined with a shortened enzyme name. Consider manually assigning a shorter name for this enzyme or its associated rxns.')
		shortened_enz_name = enz[:max_rxn_id_length]
	if rxns_with_only_this_enz:
		# print(f'Enzyme {enz} is only associated with rxns {rxns_with_only_this_enz}, so using shortest unique rxn name {best_rxn_id} as base for enzyme name.')
		# change the key enz_rxn_dict_updated to best_rxn_id
		update_GPR_enz_dict(enz, best_rxn_id)
		enz_rxn_dict_updated[best_rxn_id] = enz_rxn_dict_updated.pop(enz)
	else:
		shortened_enz_name = best_rxn_id
		# try to find shorter name by removing characters from enzyme name while keeping it unique
		new_base_name = best_rxn_id + 'ENZ_'
		# print(f'Warning: combined length of enzyme name {enz} and shortest rxn name {best_rxn_id} is {len(best_rxn_id+enz)}, which is longer than the maximum allowed length of {max_rxn_id_length}. Consider manually assigning a shorter name for this enzyme.')
		if len(new_base_name) > max_rxn_id_length:
			warn(f'Even the shortest rxn name {best_rxn_id} combined with a shortened enzyme name is too long for GAMS. Consider manually assigning a shorter name for this enzyme.')
		# for each rxn it catalyzes, find all subunits that aren't present in every other izozyme for that rxn, and see if including them in the enzyme name would make it unique and informative while fitting within GAMS limits
		for rxn in rxns:
			other_enzs_for_rxn = [e for e in enz_rxn_dict if rxn in enz_rxn_dict[e]['rxns'] and e != enz]
			subunits = set(enz_rxn_dict[enz]['protein_stoich'].keys())
			universal_subunits = set()
			for other_enz in other_enzs_for_rxn:
				subunits.update(enz_rxn_dict[other_enz]['protein_stoich'].keys())
				if not universal_subunits:
					universal_subunits = set(enz_rxn_dict[other_enz]['protein_stoich'].keys())
				else:
					universal_subunits.intersection_update(enz_rxn_dict[other_enz]['protein_stoich'].keys())
			distinctive_subunits = set(enz_rxn_dict[enz]['protein_stoich'].keys()) - universal_subunits
			# if combining these subunits with the best_rxn_id would make the enzyme name unique, informative, and short enough, use that as the new enzyme name
			test_name = new_base_name + ''.join(sorted(distinctive_subunits))
			if len(test_name) <= max_rxn_id_length and test_name not in enz_rxn_dict_updated:
				update_GPR_enz_dict(enz, test_name)
				enz_rxn_dict_updated[test_name] = enz_rxn_dict_updated.pop(enz)
				break
			# among all distinctive subunits, find shortest substrings that make them unique within the set of subunits, and see if combining those with the best_rxn_id would make the enzyme name unique, informative, and short enough
			shortened_subunits = find_shortest_unique_substring(subunits)
			shortened_distinctive_subunits = {s:shortened_subunits[s] for s in distinctive_subunits}
			test_name = new_base_name + '_'.join(sorted(shortened_distinctive_subunits.values()))
			if len(test_name) <= max_rxn_id_length and test_name not in enz_rxn_dict_updated:
				update_GPR_enz_dict(enz, test_name)
				enz_rxn_dict_updated[test_name] = enz_rxn_dict_updated.pop(enz)
				break

		else:
			warn(f'Could not find a way to shorten enzyme name for {enz} that is informative and fits within GAMS limits. Consider manually assigning a shorter name for this enzyme or its associated rxns.')

if len(enz_rxn_dict_updated) != len(enz_rxn_dict):
	raise ValueError(f'Warning: the number of enzymes changed after updating names to fit GAMS limits. Original number of enzymes: {len(enz_rxn_dict)}, new number of enzymes: {len(enz_rxn_dict_updated)}. This may indicate that some enzymes were merged due to having the same shortened name. Consider reviewing the updated enzyme names for potential issues.')

enz_rxn_dict = enz_rxn_dict_updated

# give each enz in enz_rxn_dict a row in df_enz_rxns
new_rows = []
for enz in enz_rxn_dict:
	# only add enzymes that are not already in df_enz_rxns
	if df_enz_rxns[df_enz_rxns.enz == enz].empty:
		# calculate MW of each enzyme based on its protein stoichiometry
		if mw in enz_rxn_dict[enz] and enz_rxn_dict[enz][mw]:
			enz_mw = enz_rxn_dict[enz][mw]
		else:
			enz_mw = 0
			for prot in enz_rxn_dict[enz]['protein_stoich']:
				pro_mw = df_pro.loc[prot,mw] if prot in df_pro.index else 0
				# check in gene_src column
				if pro_mw == 0:
					if prot in df_pro.gene_src.to_list():
						# print(prot,df_pro.loc[df_pro.gene_src == prot,mw])
						# if there are multiple proteins with the same gene_src, check if their MW is the same
						# if not, print a warning
						# pick the first one
						pro_match = df_pro.loc[df_pro.gene_src == prot,mw].to_list()
						if df_pro.loc[df_pro.gene_src == prot,mw].nunique() > 1:
							raise ValueError(f'Warning: {prot} has multiple MWs in df_pro')
						if df_pro.loc[df_pro.gene_src == prot,'sequence'].nunique() > 1:
							raise ValueError(f'Warning: {prot} has multiple sequences in df_pro')
						pro_mw = pro_match[0]
					else:
						warn(f'{prot} not found in df_pro')
				enz_mw += pro_mw*enz_rxn_dict[enz]['protein_stoich'][prot]
		if enz_mw == 0:
			warn(f'MW of enzyme {enz} is 0')
		if (not ignore_MW_zeros) or (enz_mw > 0):
			new_rows.append({'enz':enz,'rxns':','.join(enz_rxn_dict[enz]['rxns']),
							'protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']]),
							mw:enz_mw,
							'status':'automatically added'})
		# print(f'Adding {enz} with {enz_rxn_dict[enz]["rxns"]} and MW {enz_mw}')
	# check if cofactors for any subunits are used in rxns (safer lookups to avoid KeyError)
	cofactor_dict = {}
	for prot in enz_rxn_dict[enz]['protein_stoich']:
		# safely retrieve Cofactor info from df_pro (index may be id while prot is gene_src)
		cof_field = None
		if prot in df_pro.index and 'Cofactor' in df_pro.columns:
			cof_field = df_pro.at[prot, 'Cofactor']
		elif 'gene_src' in df_pro.columns:
			pro_matches = df_pro.loc[df_pro['gene_src'] == prot, 'Cofactor']
			if not pro_matches.empty:
				# use the first match (warn if multiple)
				if pro_matches.nunique() > 1:
					print(f"Warning: multiple Cofactor entries for gene_src {prot}; using the first.")
				cof_field = pro_matches.iloc[0]
		# if we have a cof_field, extract CHEBI ids
		if pd.notna(cof_field) and cof_field:
			for cofactor in set(re.findall(r'CHEBI:(\d+)', str(cof_field))):
				cofactor_dict.setdefault(cofactor, []).append(prot)

	if cofactor_dict:
		# print(f'cofactor dict for {enz}: {cofactor_dict}')
		unique_cofactors = cofactor_dict.keys()
		# compare to stoichiometry of all rxns using this enzyme
		met_rxns_dict = {}
		for rxn in enz_rxn_dict[enz]['rxns']:
			for met in model.reactions.get_by_id(rxn).metabolites:
				ann = getattr(met, 'annotation', {}) or {}
				if 'chebi' in ann and ann.get('chebi'):
					# normalize to list
					chebi_vals = ann.get('chebi').replace('CHEBI:','')
					if not isinstance(chebi_vals, list):
						chebi_vals = [chebi_vals]
					met_rxns_dict.setdefault(met, []).extend(chebi_vals)
		# Now check which cofactors appear among metabolite annotations and add comments safely
		for cofactor, prots in cofactor_dict.items():
			# find metabolites that annotate this cofactor
			used_mets = [met for met, chebis in met_rxns_dict.items() if any(str(c) == str(cofactor) for c in chebis)]
			if not used_mets:
				continue
			used_rxn_ids = []
			for met in used_mets:
				# collect representative rxn ids (metabolite -> reactions using it)
				used_rxn_ids.extend([r.id for r in met.reactions if r.id in enz_rxn_dict[enz]['rxns']] if hasattr(met, 'reactions') else [])
			used_rxn_ids = sorted(set(used_rxn_ids))
			for pro in prots:
				# find matching index/rows in df (df and df_pro are the same object earlier, but indices can differ)
				if pro in df.index:
					target_idxs = [pro]
				elif 'gene_src' in df.columns:
					target_idxs = df.index[df['gene_src'] == pro].tolist()
				else:
					target_idxs = []
				if not target_idxs:
					print(f"Warning: cannot add cofactor comment for {pro} — protein not found in dataframe.")
					continue
				for idx in target_idxs:
					msg = f"Cofactor may be redundant, as it appears in rxns {used_rxn_ids}"
					# ensure 'comments' column exists and is a string
					cur = df.at[idx, 'comments'] if ('comments' in df.columns and pd.notna(df.at[idx, 'comments'])) else ''
					if msg not in cur:
						df.at[idx, 'comments'] = (cur + ' ' + msg).strip()

df_enz_rxns = pd.concat([df_enz_rxns, pd.DataFrame(new_rows)], sort=False, ignore_index=True)
# sort by id, MW, and then by rxns
df_enz_rxns = df_enz_rxns.sort_values(by=['rxns',mw,'enz'])
# save automatically updated df_enz_rxns as separate file
os.makedirs('model', exist_ok=True)
df_enz_rxns.to_csv('input/ENZYME_stoich.tsv', sep='\t', index=False)

Reactions to ignore when checking if enzymes must be added:
['EX_arab__D_e', 'EX_csn_e', 'EX_cytd_e', 'EX_fru_e', 'EX_nmn_e', 'EX_dad_2_e', 'EX_duri_e', 'EX_din_e', 'EX_dgsn_e', 'TRDRDt_c_x', 'FALDH_c', 'ACCAH_x', 'FBP2_c', 'LYSTRS_m', 'EX_ins_e', 'EX_ddca_e', 'EX_lanost_e', 'ZYM181t_l_rm', 'EAR80y_m', 'EAR60y_m', 'EX_pydxn_e', 'EX_gsn_e', 'HISTRS_m', 'PNP_c', 'PUNP1_c', 'PUNP1_m', 'FECPOR_m', 'NADK_m', 'BHMT_n', 'ALCD19y_c', 'OHPHM_m', 'DOLK_c', 'VALTRS_m', 'ACHLE3_c', 'URO_x', '4H2OGLTt_c_x', 'EX_etac_e', 'EX_etha_e', 'EX_srb__L_e', 'ZYM161t_l_rm', 'LANOST181t_l_rm', 'EPIST161t_l_rm', 'LANOST161t_l_rm', 'FECOST181t_l_rm', 'FECOST161t_l_rm', 'EPIST181t_l_rm', 'EX_2mbac_e', 'EX_sucr_e', 'EX_hxdcal_e', 'GDPPAILMT_rm', 'EX_13BDglucan_e', 'EX_crn_e', 'PUNP7_c', 'NADHK1_m', 'NAMNPP_m', 'DAPO_c', 'MHPGLUT_c', 'EX_8aonn_e', 'EX_5fthf_e', 'TCHOLAabc_v', 'MN2tabc_rm', 'EX_epist_e', 'METt_c_m', 'PHEACt_c_e', 'EX_amet_e', 'STEHz181_en', 'EX_4abut_e', 'EX_gthox_e', 'EX_gthrd_e', 'ACONTa_c', 'ASPt

/var/folders/fx/q_x2rm651s91fj2m6_tb19pxvpxbmb/T/ipykernel_4547/675813401.py:351: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_enz_rxns = pd.concat([df_enz_rxns, pd.DataFrame(new_rows)], sort=False, ignore_index=True)


In [30]:
# check if all rxns in model are accounted for in df_enz_rxns, and if they have been automatically updated
# make a list of all rxns in df_enz_rxns
enz_rxns = []
for index, row in df_enz_rxns.iterrows():
	enz_rxns += row['rxns'].split(',')
enz_rxns = sorted(list(dict.fromkeys(enz_rxns)))
# check if all rxns in model are in enz_rxns
missing_rxns = []
for rxn in model.reactions:
	if rxn.id not in rxns_not_needing_enzymes and rxn.id not in enz_rxns and rxn.id not in whole_biomass_rxns and rxn.gene_reaction_rule not in spont_rxn_dict.keys():
		missing_rxns.append(rxn.id)
if len(missing_rxns) > 0:
	warn(f'the following reactions are not accounted for in df_enz_rxns: {missing_rxns}')
# remove rxns from rxn_updates if their value is {}
for rxn in rxn_updates.copy():
	if not rxn_updates[rxn]:
		del rxn_updates[rxn]
	else:
		print(f'Updating {rxn} with {rxn_updates[rxn]}')
# if rxn_updates isn't empty, print a warning
# if len(rxn_updates) > 0:
# 	raise ValueError(f'Warning: the following reactions have been automatically updated:\n{rxn_updates}')
# convert df_enz_rxns to old ENZYME_stoich_curation file format from 2024 version of scRBA, for compatibility

df_rxn_enz_pairs = pd.DataFrame(columns=['id','rxn_src','enz','protein_stoich'])
catalyst_rxn_dict = {**spont_rxn_dict, **enz_rxn_dict}
for k, v in catalyst_rxn_dict.items():
	if k in spont_rxn_dict:
		catalyst_rxn_dict[k]['rxns'].extend(spont_rxn_dict[k]['rxns']) # adds rxns that were overrided by merger with enz_rxn_dict
		catalyst_rxn_dict[k]['rxns'] = list(dict.fromkeys(catalyst_rxn_dict[k]['rxns']))
for enz in catalyst_rxn_dict:
	enz_id = enz if enz != '' else 'SPONT'
	# don't seem to need MW of enzyme in prior RBA code
	prot_stoich_str = 'zeroCost' if enz in RBA_defaults.spont_GPRs + [''] else None
	rxn_compartment_pairs = {r: sorted(model.reactions.get_by_id(r).compartments) for r in catalyst_rxn_dict[enz]['rxns']}
	all_rxn_compartments = set()
	for comps in rxn_compartment_pairs.values():
		all_rxn_compartments.update(comps)
	# don't seem to need MW of enzyme in prior RBA code
	for rxn in catalyst_rxn_dict[enz]['rxns']:
		dirs = []
		update_enz_id = False
		compartments_to_add = set()
		min_heteromer_term = None # for comparing UniProt to GSM stoichiometries
		if prot_stoich_str != 'zeroCost':
			prot_stoich_str = ''
			rxn_compartments = rxn_compartment_pairs[rxn]
			for prot in catalyst_rxn_dict[enz]['protein_stoich']:
				prot_id = None
				if prot in df_pro.index and df_pro.loc[prot,'subloc_assigned'] in rxn_compartments:
					prot_id = prot
				if not prot_id:
					update_enz_id = True
					# check gene_src column
					prot_matches = df_pro[df_pro.gene_src == prot]
					if not prot_matches.empty:
						# if multiple matches, check if any have subloc_assigned in rxn_compartments
						for idx, match in prot_matches.iterrows():
							if match['subloc_assigned'] in rxn_compartments:
								prot_id = match['id']
								break
						# if still not found, pick the first one and print a warning
						if not prot_id:
							if len(prot_matches) > 1 or prot in proteins_with_location_conflicts: # important to check if already in conflicts, since conflict resolution happens after this step and the prior conflict checks are less comprehensive, which affects how they're handled (e.g., adding compartment if only 1 conflict detected)
								if len(prot_matches) > 1:
									# use compartment_adjacency_map to find closest compartment, with compartments inside rxn_compartments preferred to ones outside
									possible_comps = []
									for idx, match in prot_matches.iterrows():
										if pd.notna(match['subloc_assigned']):
											possible_comps += match['subloc_assigned'].split(',')
									possible_comps = list(dict.fromkeys(possible_comps))
									protein_rxn_adjacent_comps = {'inside':[], 'outside':[]}
									for comp in possible_comps:
										for rxn_comp in rxn_compartments:
											if rxn_comp in compartment_adjacency_map.get(comp, []):
												protein_rxn_adjacent_comps['outside'].append(comp)
											elif comp in compartment_adjacency_map.get(rxn_comp, []):
												protein_rxn_adjacent_comps['inside'].append(comp)
									msg = f" Protein {prot} not in reaction compartments {rxn_compartments}."
									if protein_rxn_adjacent_comps['inside']:
										prot_matches = prot_matches[prot_matches['subloc_assigned'].isin(protein_rxn_adjacent_comps['inside'])]
										msg += f" Present in compartments inside them: {protein_rxn_adjacent_comps['inside']}."
									if protein_rxn_adjacent_comps['outside']:
										if not protein_rxn_adjacent_comps['inside']:
											prot_matches = prot_matches[prot_matches['subloc_assigned'].isin(protein_rxn_adjacent_comps['outside'])]
											msg += f" Present in compartments outside them: {protein_rxn_adjacent_comps['outside']}."
									if protein_rxn_adjacent_comps['inside'] or protein_rxn_adjacent_comps['outside']:
										prot_id = prot_matches.iloc[0]['id']
										cp_map_col = 'compartment-protein_maps'
										enz_mask = df_enz_rxns['enz'] == enz
										enz_idx = df_enz_rxns[enz_mask].index[0]
										chosen_comp = prot_matches.iloc[0]['subloc_assigned']
										# if only 1 compartment is used in rxns for this enzyme in df_enz_rxns, modify protein_stoich to use prot_id instead of prot
										if len(all_rxn_compartments) == 1:
											# update compartment-protein_maps in df_enz_rxns to include rxn compartment as key and dict with prot as key and prot_id as value
											if pd.isna(df_enz_rxns.at[enz_idx, cp_map_col]):
												df_enz_rxns.at[enz_idx, cp_map_col] = {chosen_comp: {prot: prot_id}}
											else:
												df_enz_rxns.at[enz_idx, cp_map_col].setdefault(chosen_comp, {})[prot] = prot_id
										# if there are no isozymes for this rxn, or all require this protein, add the compartment automatically since no other protein is an obvious alternative
										elif all(
												prot in catalyst_rxn_dict[other_enz]['protein_stoich']
												for other_enz in GPR_enz_dict.get(model.reactions.get_by_id(rxn).gene_reaction_rule, [])
										):
											# if there are multiple rxn compartments, only add the one used by most metabolites in the rxn
											if len(rxn_compartments) > 1:
												comp_count = {comp:0 for comp in rxn_compartments}
												for met in model.reactions.get_by_id(rxn).metabolites:
													met_comp = str(met.compartment)
													if met_comp in comp_count:
														comp_count[met_comp] += 1
												sorted_comps = sorted(comp_count.items(), key=lambda x: x[1], reverse=True)
												print(f'Compartment counts for metabolites in reaction {rxn}: {comp_count}')
												chosen_comp = sorted_comps[0][0]
												print(f'Automatically adding compartment {chosen_comp} to protein {prot} due to its necessity for reaction {rxn}, chosen among multiple compartments based on metabolite presence.')
											else:
												chosen_comp = list(rxn_compartments)[0]
											# print(f'Automatically adding compartment {chosen_comp} to protein {prot} due to its necessity for reaction {rxn}.')
											# update compartment-protein_maps in df_enz_rxns to include rxn compartment as key and dict with prot as key and prot_id as value
											if pd.isna(df_enz_rxns.loc[enz_idx, cp_map_col]):
												df_enz_rxns.at[enz_idx, cp_map_col] = {chosen_comp: {prot: prot_id}}
											else:
												df_enz_rxns.at[enz_idx, cp_map_col].setdefault(chosen_comp, {})[prot] = prot_id
										# if it's between compartments for this rxn, may be a transporter or transmembrane protein
										else:
											print(f'Note: protein {prot} mapped to {prot_id} for reaction {rxn} based on compartment adjacency.')
											proteins_with_location_conflicts.setdefault(prot, set()).update(rxn_compartments)
									else:
										# only treat as a conflict if no adjacent compartments found
										proteins_with_location_conflicts.setdefault(prot, set()).update(rxn_compartments)
									# print if multiple outside compartments found
									if len(prot_matches) > 1:
										msg += f" Multiple possible compartments found: {possible_comps}."
									print(f'Warning:{msg}')
							prot_id = prot_matches.iloc[0]['id']
				if prot_id:
					heteromer_term = get_mer(df_pro.loc[prot_id],required_prefixes=['hetero'],find_for_whole_complex=True)
					if heteromer_term not in (None, 1): # excluding '1' since it's printed by default
						if min_heteromer_term is None:
							min_heteromer_term = heteromer_term
						else:
							min_heteromer_term = min(min_heteromer_term, heteromer_term)
					if catalyst_rxn_dict[enz]['protein_stoich'].get(prot) is None:
						prot_stoich_str += f"{prot_id}:{catalyst_rxn_dict[enz]['protein_stoich'][prot_id]},"
					else:
						prot_stoich_str += f"{prot_id}:{catalyst_rxn_dict[enz]['protein_stoich'][prot]},"
					compartments_to_add.add(df_pro.at[prot_id,'subloc_assigned'])
				else:
					print(f'Warning: Protein {prot} not found in df_pro for reaction {rxn} with compartments {rxn_compartments}.')
			# remove trailing comma
			prot_stoich_str = prot_stoich_str.rstrip(',')
		
		enz_id_final = enz_id
		if update_enz_id:
			# update enz_id to include sublocations
			enz_id_final = '_'.join([enz_id] + sorted(list(compartments_to_add)))
		# add FWD and REV to rxn_src if those directions are possible
		if model.reactions.get_by_id(rxn).lower_bound < 0:
			dirs.append('REV')
		if model.reactions.get_by_id(rxn).upper_bound > 0:
			dirs.append('FWD')
		for dir in dirs:
			# print(enz,rxn)
			# df_rxn_enz_pairs = df_rxn_enz_pairs.append({'id':f'RXN-{rxn}_{dir}-{enz}','enz':enz,'gpr':'','protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']]),'status':'automatically added'}, ignore_index=True)
			df_rxn_enz_pairs.loc[len(df_rxn_enz_pairs)] = [f'RXN-{rxn}_{dir}-{enz_id_final}',rxn,enz_id_final,prot_stoich_str]
			# check if total protein stoich compares to heteromer terms (to identify missing subunits and other issues)
			if prot_stoich_str != 'zeroCost':
				enz_stoich_subunits_total = sum(catalyst_rxn_dict[enz]['protein_stoich'].values())
				if min_heteromer_term not in (None, 1) and enz_stoich_subunits_total != min_heteromer_term:
					warn(f'Enzyme {enz} for reaction {rxn} has {enz_stoich_subunits_total} subunits, while Uniprot subunit structure predicts {min_heteromer_term} subunits.')
			# df_rxn_enz_pairs = pd.concat([df_rxn_enz_pairs,pd.DataFrame({'id':f'RXN-{rxn}_{dir}-{enz}','enz':enz,'protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']])})], ignore_index=True)

# sort by ID
df_rxn_enz_pairs = df_rxn_enz_pairs.sort_values(by='id')

for prot in proteins_with_location_conflicts:
	# if there's only 1 compartment, add it to the protein's subloc_assigned
	if len(proteins_with_location_conflicts[prot]) == 1:
		comp = list(proteins_with_location_conflicts[prot])[0]
		pro_match = df.loc[df['gene_src'] == prot]
		if pro_match.empty:
			pro_match = df.loc[df.index == prot]
		# if still empty, skip
		if pro_match.empty:
			warn(f'protein {prot} not found in df to resolve location conflict.')
			continue
		if pd.notna(df.at[prot,'subloc_assigned']):
			sublocs = df.at[prot,'subloc_assigned'].split(',')
		else:
			sublocs = []
		if comp not in sublocs:
			# check if compartment_adjacency_map has any compartments close to comp that are already in sublocs
			current_adjacent_comps = {'inside':set(), 'outside':set()}
			for subloc in sublocs:
				if comp in compartment_adjacency_map[subloc]:
					current_adjacent_comps['inside'].add(subloc)
				if subloc in compartment_adjacency_map[comp]:
					current_adjacent_comps['outside'].add(subloc)
			msg_parts = []
			if current_adjacent_comps['inside']:
				msg_parts.append(f"inside some of its current compartments ({', '.join(current_adjacent_comps['inside'])})")
			if current_adjacent_comps['outside']:
				msg_parts.append(f"outside some of its current compartments ({', '.join(current_adjacent_comps['outside'])})")
			sublocs.append(comp)
			df.at[prot,'subloc_assigned'] = ','.join(sublocs)
			# remove from df_enz_rxns compartment-protein_maps if present
			for enz_idx, row in df_enz_rxns.iterrows():
				cp_map = row['compartment-protein_maps']
				if pd.notna(cp_map) and isinstance(cp_map, dict):
					for c in list(cp_map.keys()):
						if prot in cp_map[c]:
							del df_enz_rxns.at[enz_idx, 'compartment-protein_maps'][c][prot]
							# if that compartment now has no proteins mapped, remove the compartment entry
							if not df_enz_rxns.at[enz_idx, 'compartment-protein_maps'][c]:
								del df_enz_rxns.at[enz_idx, 'compartment-protein_maps'][c]
							# if compartment-protein_maps is now empty, make empty
							if not df_enz_rxns.at[enz_idx, 'compartment-protein_maps']:
								df_enz_rxns.at[enz_idx, 'compartment-protein_maps'] = pd.NA
			print(f'Automatically added compartment {comp} to protein {prot} subloc_assigned due to location conflict resolution.')
			if msg_parts:
				print(f" (note: {comp} is {' and '.join(msg_parts)})")
	else:
		warn(f'Protein {prot} has location conflicts with multiple compartments: {proteins_with_location_conflicts[prot]}')

df_rxn_enz_pairs.to_csv('input/ENZYME_rxn_pairs.tsv', sep='\t', index=False)
df_enz_rxns.to_csv('input/ENZYME_stoich.tsv', sep='\t', index=False)

# save df
df.to_csv(protein_stoich_path, index=False, sep='\t')

Updating lumpFACS160_c with {'enz': 'JL09_g3954JL09_g50'}
Updating lumpFACS180_c with {'enz': 'JL09_g3954JL09_g50'}
Updating ARABDt_c with {'gene_reaction_rule': ''}
Updating CYTK2_c with {'enz': 'JL09_g2829'}
Updating CYTK1_c with {'enz': 'JL09_g2829'}
Updating COAt_c_x with {'gene_reaction_rule': ''}
Updating E4Pt_c_m with {'gene_reaction_rule': ''}
Updating EX_nac_e with {'gene_reaction_rule': ''}
Updating NCAMt_c_n with {'gene_reaction_rule': ''}
Updating NH4t_c_m with {'gene_reaction_rule': ''}
Updating NADPHt_c_r with {'gene_reaction_rule': ''}
Updating NADPt_c_r with {'gene_reaction_rule': ''}
Updating MI1Pt_c_r with {'gene_reaction_rule': ''}
Updating TYMR_c with {'enz': 'JL09_g1388'}
Updating MI13456Pt_c_n with {'gene_reaction_rule': ''}
Updating 2DDA7Pt_c_m with {'gene_reaction_rule': ''}
Updating 2DHPt_c_m with {'gene_reaction_rule': ''}
Updating PMDPHT_c with {'gene_reaction_rule': ''}
Updating PYNP2r_c with {'gene_reaction_rule': ''}
Updating PPIt_c_m with {'gene_reaction_

TypeError: sequence item 1: expected str instance, float found

In [ ]:
# # find all reactions that weren't added to specific compartments but potentially should have been
# rxn_stoichs = {}
# mets_to_check_rxns_for = set()
# rxns_to_check = []
# # Prepare a set of unique metabolite IDs and a mapping from ID to dict version
# unique_met_ids = set(met.id for met in unique_mets)
# unique_mets_dict_by_id = {met['id']: met for met in unique_mets_dict}

# for r in model.reactions:
# 	# ignore blocked reactions (to avoid unnecessary work from adding them)
# 	if r.bounds == (0,0):
# 		continue
# 	# ignore non-spontaneous reactions (since their proteins may not localize to the compartment)
# 	if not (r.genes is None or len(r.genes) == 0 or all([g.id in genes_without_protein_id for g in r.genes])): # i.e., is non-spontaneous
# 		continue
# 	# ignore transport and exchange reactions (and pseudoreactions, since they're generally manually curated)
# 	if r.subsystem.lower() in ['transport','transport, extracellular','exchange','pseudoreaction'] or r.id in model.exchanges:
# 		continue
# 	# ignore biomass reactions
# 	if r.id in SM_biomass_rxns + whole_biomass_rxns:
# 		continue
# 	# add to rxns_to_check
# 	# rxns_to_check.append(r.id)
# 	if len(r.compartments) > 1:
# 		print(f'Warning: {r.id} has multiple compartments: {r.compartments}, skipping this reaction for compartment checking')
# 		compartments_with_all_mets = None
# 	else:
# 		# make set of compartments to check (exclude compartments used by this reaction)
# 		compartments_with_all_mets = set(met.compartment for met in r.metabolites)
# 	compartments_to_check = set(model.compartments.keys()) - set(met.compartment for met in r.metabolites)
# 	if not compartments_to_check:
# 		# if there are no compartments to check, skip this reaction
# 		continue
# 	# add to rxn_stoichs, using unique_mets to standardize metabolite IDs across compartments
# 	rxn_stoich_dict = {}
# 	for met, coef in r.metabolites.items():
# 		found = False
# 		universal_id = None
# 		if met.id in unique_met_ids:
# 			rxn_stoich_dict[met.id] = coef
# 			# Check for 'version_diffs' in the dict version
# 			if 'version_diffs' in unique_mets_dict_by_id.get(met.id, {}):
# 				mets_to_check_rxns_for.add(met.id)
# 			found = True
# 			universal_id = met.id
# 		if not found:
# 			# find in version_diffs of unique_mets_dict_by_id
# 			for um_id, um_dict in unique_mets_dict_by_id.items():
# 				if 'version_diffs' in um_dict and met.id in um_dict['version_diffs']:
# 					rxn_stoich_dict[um_id] = coef
# 					found = True
# 					universal_id = um_id
# 					mets_to_check_rxns_for.add(um_id)
# 					break
# 		if not found:
# 			# remove rxn since it has a met that isn't in any other compartments
# 			print(f'Warning: {met.id} in {r.id} not found in unique_mets, skipping this reaction for compartment checking')
# 			# rxns_to_check.remove(r.id)
# 			break
# 		else:
# 			compartments_with_met = set()
# 			compartments_with_met.add(unique_mets_dict_by_id[universal_id]['compartment'])
# 			# if unique_mets has a 'version_diffs' field for this met, add those compartments to compartments_with_met
# 			if 'version_diffs' in unique_mets_dict_by_id.get(universal_id, {}):
# 				for vdiff in unique_mets_dict_by_id[universal_id]['version_diffs']:
# 					# get compartment from vdiff (the part after the last underscore)
# 					if 'compartment' not in unique_mets_dict_by_id[universal_id]['version_diffs'][vdiff]:
# 						comp = vdiff.rsplit('_',1)[-1]
# 						print(f"Warning: 'compartment' not found in version_diffs for {universal_id} {vdiff}, assuming compartment is {comp} based on ID")
# 					else:
# 						comp = unique_mets_dict_by_id[universal_id]['version_diffs'][vdiff]['compartment']
# 					compartments_with_met.add(comp)
# 			# intersect compartments_to_check with compartments_with_met
# 			compartments_to_check = compartments_to_check.intersection(compartments_with_met)
# 	else: 
# 		# include rxn direction in the stoichiometry by adding a suffix to the reaction ID
# 		dir = 'BOTH'
# 		if r.lower_bound < 0:
# 			dir = 'REV'
# 		else:
# 			dir = 'FWD'
		
# 		# standardize stoich. direction so the metabolite that appears 1st alphabetically has a coefficient of -1
# 		min_met_id = min(rxn_stoich_dict.keys())
# 		if rxn_stoich_dict[min_met_id] < 0:
# 			rxn_stoich_dict = {met_id: -coef for met_id, coef in rxn_stoich_dict.items()}
# 			# change rxn_id suffix accordingly
# 			if dir == 'FWD':
# 				dir = 'REV'
# 			elif dir == 'REV':
# 				dir = 'FWD'
# 		rxn_stoich_dict['__dir__'] = dir
# 		# sort rxn_stoich_dict by keys to ensure consistent ordering
# 		rxn_stoich_dict = dict(sorted(rxn_stoich_dict.items()))
# 		# add to rxn_stoichs using a hashable key, if not already present
# 		if frozenset(rxn_stoich_dict.items()) in rxn_stoichs:
# 			# check if any compartments in compartments_to_check are not already represented by another reaction with the same stoichiometry
# 			existing_rxns = rxn_stoichs[frozenset(rxn_stoich_dict.items())]
# 			existing_comps = set()
# 			for ex_rxn in existing_rxns:
# 				for met in model.reactions.get_by_id(ex_rxn).metabolites:
# 					existing_comps.add(met.compartment)
# 			compartments_to_check = compartments_to_check - existing_comps
# 			if not compartments_to_check:
# 				# if all compartments are already represented, skip adding this reaction
# 				continue
# 		if compartments_to_check:
# 			# if compartments_to_check is not empty, this reaction may need to be added to those compartments
# 			print(f'{r.id} may need to be added to compartments: {compartments_to_check}')
# 			rxns_to_check.append(r.id)
# 			rxn_stoichs.setdefault(frozenset(rxn_stoich_dict.items()), []).append(r.id)

# print(f'Checking the following {len(rxns_to_check)} reactions for potential compartment additions: {sorted(rxns_to_check)}')

# # find rxn_stoichs keys with multiple values
# if any(len(rxn_stoichs[stoich]) > 1 for stoich in rxn_stoichs):
# 	print('The following reactions share the same stoichiometry:')
# 	for stoich in rxn_stoichs:
# 		if len(rxn_stoichs[stoich]) > 1:
# 			for rxn in rxn_stoichs[stoich]:
# 				print(f'\t{rxn}: {model.reactions.get_by_id(rxn).build_reaction_string()}')

# # while rxns added by looping over stoich. is not 0, keep looping

# # find all unique rxn stoichiometries, using unique_mets and model_simplified
# # unique_stoichs = {}
# # for rxn in model.reactions:
# # 	if rxn.id in rxns_not_needing_enzymes:
# # 		continue
# # 	# get the stoichiometry of the reaction as a dict of met IDs and their coefficients
# # 	stoich = {met.id: coeff for met, coeff in rxn.metabolites.items() if met.id in unique_mets}
# # 	# convert to a frozenset of tuples to make it hashable
# # 	stoich_frozenset = frozenset(stoich.items())
# # 	if stoich_frozenset not in unique_stoichs:
# # 		unique_stoichs[stoich_frozenset] = {'rxns': [rxn.id], 'stoich': stoich}
# # 	else:
# # 		unique_stoichs[stoich_frozenset]['rxns'].append(rxn.id)

In [ ]:
# apply manual changes from 01a_organism-specific_changes.py if present, to allow faster curation
manual_changes_path = 'input/01a_organism-specific_changes.py'
if os.path.exists(manual_changes_path):
	exec(open(manual_changes_path).read())

KeyError: 'JL09_g3328JL09_g3622JL09_g4214'

In [ ]:
# check if any proteins have identical sequences (i.e., are duplicates), once removing any special characters (e.g., '*', 'X', etc.) from the sequences. If so, raise a warning.
seqs = df_pro['sequence'].str.replace(r'[^A-Za-z]', '', regex=True)
if seqs.duplicated().any():
	dup_seqs = seqs[seqs.duplicated(keep=False)]
	dup_proteins = df_pro[df_pro['sequence'].isin(dup_seqs)]['id'].tolist()
	warn(f"Duplicate protein sequences found for proteins: {dup_proteins}. Consider removing duplicates or merging them into a single protein entry.")
# check if any protein's sequence is a substring of another protein's sequence. If so, raise a warning.
for i, seq1 in enumerate(seqs):
	for j, seq2 in enumerate(seqs):
		if i != j and seq1 in seq2:
			pro1 = df_pro.index[i]
			pro2 = df_pro.index[j]
			warn(f"Protein {pro1}'s sequence is a substring of protein {pro2}'s sequence. Consider merging or removing one of them, or treating as homologs if appropriate.")

In [ ]:
# remove compartment-protein_map entries that are now unnecessary after the manual changes
df = df_pro.copy()
for enz_idx, row in df_enz_rxns.iterrows():
	cp_map = row['compartment-protein_maps']
	if pd.notna(cp_map) and isinstance(cp_map, dict):
		for c in list(cp_map.keys()):
			proteins_in_comp = list(cp_map[c].keys())
			# check if all proteins in this compartment are already assigned to this compartment in df_pro
			all_assigned = True
			for pro in proteins_in_comp:
				pro_match = df.loc[df['gene_src'] == pro]
				if pro_match.empty:
					pro_match = df.loc[df.index == pro]
				# if still empty, skip
				if pro_match.empty:
					all_assigned = False
					break
				sublocs = pro_match.iloc[0]['subloc_assigned']
				if pd.isna(sublocs) or c not in sublocs.split(','):
					print(f'Protein {pro} not assigned to compartment {c} in df_pro {sublocs}, keeping compartment-protein_map entry.')
					all_assigned = False
					break
			if all_assigned:
				# remove this compartment entry
				del df_enz_rxns.at[enz_idx, 'compartment-protein_maps'][c]
		# if compartment-protein_maps is now empty, make empty
		if not df_enz_rxns.at[enz_idx, 'compartment-protein_maps']:
			df_enz_rxns.at[enz_idx, 'compartment-protein_maps'] = pd.NA
# save updated enzyme dataframe
df_enz_rxns.to_csv('input/ENZYME_stoich.tsv', sep='\t', index=False)

Protein JL09_g3902 not assigned to compartment m in df_pro c,x,mm, keeping compartment-protein_map entry.
Protein JL09_g3078 not assigned to compartment m in df_pro c,mm,x, keeping compartment-protein_map entry.
Protein JL09_g1663 not assigned to compartment rm in df_pro c,r,l, keeping compartment-protein_map entry.
Protein JL09_g944 not assigned to compartment m in df_pro mm,c,r, keeping compartment-protein_map entry.


In [ ]:
if warnings_summary:
	print('Warnings encountered during processing:')
	for warning in warnings_summary:
		print(warning)
	raise Warning ("Processing completed with warnings; please review the messages above, starting under 'Warnings encountered during processing:'.")

Warnings encountered during processing:
Error querying ChEBI for ID 49883: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 24875: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 190135: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 60240: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 23378: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 38290: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 83088: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 21137: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 60342: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 49786: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 30413: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 61717: Expecting value: line 2 column 1 (char 1)
Error querying ChEBI for ID 25213: 

Warning: Processing completed with warnings; please review the messages above, starting under 'Warnings encountered during processing:'.